# Proiect A — Customer Personality Analysis (varianta Google Colab, cu explicații)

**Setul de date:** [Kaggle – imakash3011/customer-personality-analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis), fișierul `marketing_campaign.csv` — 2240 de clienți, 29 de coloane (date demografice, cheltuieli pe 2 ani pe categorii de produse, numărul de cumpărături pe canale, răspunsul la 6 campanii de marketing).

**Povestea:** un lanț de magazine pregătește o nouă campanie de oferte. Marketingul pune două întrebări:

| | Întrebarea | Ținta | Tipul problemei |
|---|---|---|---|
| 1 | Ce clienți vor accepta oferta din campanie? | `Response` (1 = a acceptat) | **clasificare binară**, clase dezechilibrate (15% / 85%) |
| 2 | Cât cheltuie un client în 2 ani? | `MntTotal` = suma celor 6 coloane `Mnt*` | **regresie** |

**De ce două probleme pe același set?** Cerința proiectului cere atât clasificare, cât și regresie. Am ales ținte care au sens de business (cui trimit oferta / cât valorează clientul), nu ținte artificiale.

---
### Cum citești notebook-ul
Fiecare pas are aceeași structură:
- 🟦 **Ce face** — pe scurt, ce calculează celula de cod;
- 🟩 **De ce așa** — motivarea alegerii;
- 🟧 **De ce nu altă variantă** — alternativele la care m-am gândit și de ce le-am respins;
- 📊 **Ce a ieșit** — rezultatele din rularea mea (după rulare le vezi și sub cod) și concluzia trasă.

### Cum rulezi în Colab
1. `File → Upload notebook` și alegi acest fișier.
2. `Runtime → Run all`. Setul de date se descarcă automat de pe Kaggle (`kagglehub`, set public, nu cere cont). Dacă descărcarea nu merge, celula 1 îți cere să încarci manual `raw.csv` (cel din `ProiectA/data/`).
3. Rularea completă durează ~20–30 de minute pe CPU; cel mai lent e capitolul 10 (rețelele Keras). GPU nu e necesar.

**Ordinea capitolelor urmează baremul:** 0 setări · 1 încărcare · 2 EDA · 3 curățare · 4 atribute + design train/test · 5 preprocesare + leakage · 6 dezechilibru · 7 modele + CV + ROC/PR · 8 grid search + senzitivitate + nested CV · 9 ansambluri + test t corectat · 10 rețele neuronale · 11 evaluarea pe test + profit · 12 concluzii.

## 0. Setări

### 0.0 Instalarea pachetelor lipsă în Colab
🟦 **Ce face:** Colab are deja `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `tensorflow`/`keras`. Lipsesc doar `imbalanced-learn` (SMOTE, oversampling în pipeline) și `kagglehub` (descărcarea setului). Celula le instalează doar dacă rulează în Colab.

🟩 **De ce:** notebook-ul trebuie să ruleze cu *Run all*, fără pași manuali.

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn", "kagglehub"], check=True)

import sklearn, matplotlib
print("Colab:", IN_COLAB, "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

### 0.1 Importuri, constante, foldere
🟦 **Ce face:** importă bibliotecile de bază, fixează `SEED = 42`, creează folderele `data/`, `figures/`, `results/`, `models/` și definește numele țintelor.

🟩 **De ce `SEED = 42` peste tot:** împărțirea train/test, fold-urile de validare, modelele cu componentă aleatoare (random forest, boosting, rețele) depind de un generator aleator. Cu același seed, oricine rulează notebook-ul obține **aceleași cifre** → rezultatele sunt reproductibile și comparațiile între modele sunt corecte (toate văd aceleași date).

🟩 **De ce constantele `TARGET_CLF`, `TARGET_REG`, `MNT_COLS`:** le folosesc în tot notebook-ul; dacă schimb ținta, o schimb într-un singur loc.

🟧 **De ce ascund `FutureWarning` / `UserWarning`:** sunt avertismente de versiune ale bibliotecilor, nu erori ale analizei; fără ele ieșirile se citesc mai ușor la prezentare.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

SEED = 42
MY_NAME = "customer"

# folderele proiectului (relative la notebook)
DATA_DIR = "data"
FIGURES_DIR = "figures"
RESULTS_DIR = "results"
MODELS_DIR = "models"
for folder in [DATA_DIR, FIGURES_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(folder, exist_ok=True)

TARGET_CLF = "Response"  # 1 = a acceptat oferta din ultima campanie
POS_CLASS = 1
TARGET_REG = "MntTotal"  # suma cheltuita in 2 ani (o construim la curatare)
MNT_COLS = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]

np.random.seed(SEED)
pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 100

### 0.2 Funcțiile folosite în tot notebook-ul
🟦 **Ce face:**
- `make_prep` — preprocesarea: 3 drumuri separate, după tipul coloanei:
  - **numerice** → completare cu **mediana** + scalare (`StandardScaler` implicit; `"passthrough"` pentru arbori);
  - **`Education`** → `OrdinalEncoder` cu ordinea dată de mine: Basic < Graduation < Master < PhD;
  - **`Marital_Status`** → `OneHotEncoder`, cu categoriile rare (< 10 clienți) unite într-o coloană „infrequent”.
- `make_pipe` — lipește preprocesarea + (opțional) selecția de atribute + modelul într-un singur `Pipeline`.
- `evaluate` / `compare` — validare încrucișată pe **aceleași** fold-uri pentru toate modelele; întorc media ± abaterea pe validare și scorul pe train (ca să văd overfitting-ul).
- `corrected_ttest` / `ttest_pair` — testul t corectat Nadeau–Bengio pentru a compara două modele.

🟩 **De ce `Pipeline` + `ColumnTransformer`:** preprocesarea (mediane, medii și abateri pentru scalare, categoriile one-hot) **se învață** din date. Dacă le-aș calcula pe tot setul înainte de împărțire, fold-urile de validare ar „scăpa” informație în antrenare (**data leakage**). În pipeline, `cross_validate` re-antrenează preprocesarea **doar pe partea de antrenare a fiecărui fold**.

🟩 **De ce mediana și nu media la completare:** `Income` are asimetrie 6,8 (câțiva clienți foarte bogați). Media e trasă în sus de valorile extreme, mediana nu.

🟩 **De ce ordinal la `Education`, dar one-hot la `Marital_Status`:** nivelurile de educație au o **ordine reală** (Basic < … < PhD) → un singur număr păstrează ordinea și nu umflă numărul de coloane. Starea civilă **nu are ordine** (Married nu e „mai mare” decât Single) → one-hot, altfel modelul ar presupune o ordine falsă.

🟧 **De ce nu `LabelEncoder` / `pd.get_dummies`:** `get_dummies` se aplică pe tot setul (în afara pipeline-ului) și nu știe ce să facă cu o categorie nouă la test; `OneHotEncoder(handle_unknown=...)` din pipeline rezolvă ambele probleme. `LabelEncoder` e pentru ținte, nu pentru atribute, și ar da o ordine alfabetică arbitrară.

🟩 **De ce metrica principală e prima din `METRICS`:** **PR-AUC** (`average_precision`) la clasificare și **MAE** la regresie — motivarea e în capitolul 7.

🟩 **De ce testul t *corectat*:** la validarea încrucișată fold-urile de antrenare se suprapun (fiecare rând apare în 4 din 5), deci scorurile pe fold-uri nu sunt independente. Testul t obișnuit subestimează varianța și declară „semnificative” diferențe care sunt doar zgomot. Corecția Nadeau–Bengio mărește varianța cu factorul `1/k + n_test/n_train`.

In [ ]:
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import cross_validate

# ordinea reala a nivelurilor de educatie (2n Cycle e unit cu Master la curatare)
EDU_ORDER = [["Basic", "Graduation", "Master", "PhD"]]


def make_prep(num_cols, scaler=None, add_indicator=False):
    # numerice: mediana (Income e foarte asimetric) + scalare; scaler="passthrough" -> fara scalare (arbori)
    num_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=add_indicator)),
                          ("scale", StandardScaler() if scaler is None else scaler)])
    # Education: ordinal, cu ordinea data de noi
    ord_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ordinal", OrdinalEncoder(categories=EDU_ORDER, handle_unknown="use_encoded_value",
                                                     unknown_value=-1))])
    # Marital_Status: one-hot; categoriile cu < 10 clienti (Other) intra intr-o coloana "infrequent"
    nom_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist",
                                                   sparse_output=False))])
    return ColumnTransformer([("num", num_steps, num_cols),
                              ("ord", ord_steps, ["Education"]),
                              ("nom", nom_steps, ["Marital_Status"])])


def make_pipe(model, num_cols, scaler=None, add_indicator=False, select=None):
    # prep (+ selectie optionala de atribute) + model
    steps = [("prep", make_prep(num_cols, scaler, add_indicator))]
    if select is not None:
        steps.append(("select", select))
    steps.append(("model", model))
    return Pipeline(steps)


# prima metrica din fiecare lista = metrica principala
METRICS = {
    "classification": {"PR_AUC": "average_precision", "ROC_AUC": "roc_auc", "F1": "f1", "recall": "recall",
                       "precision": "precision", "bal_acc": "balanced_accuracy"},
    "regression": {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"},
}


def evaluate(model, X, y, task, cv):
    # validare incrucisata; intoarce (randul de tabel, scorurile metricii principale pe fold-uri)
    scoring = METRICS[task]
    res = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    sign = {name: -1 if scorer.startswith("neg_") else 1 for name, scorer in scoring.items()}  # erorile fara minus
    main = next(iter(scoring))
    folds = sign[main] * res[f"test_{main}"]
    row = {main: folds.mean(), f"{main}_std": folds.std(), f"{main}_train": sign[main] * res[f"train_{main}"].mean()}
    row |= {name: sign[name] * res[f"test_{name}"].mean() for name in scoring if name != main}
    row["fit_time_s"] = res["fit_time"].mean()
    return row, folds


def compare(models, X, y, task, cv):
    # tabelul sortat dupa metrica principala + scorurile pe fold-uri ale fiecarui model
    rows, folds = [], {}
    for name, model in models.items():
        row, folds[name] = evaluate(model, X, y, task, cv)
        rows.append({"model": name, **row})
    main = next(iter(METRICS[task]))
    table = pd.DataFrame(rows).set_index("model").sort_values(main, ascending=(task == "regression"))
    return table, folds


def corrected_ttest(a, b, n_train, n_test):
    # testul t corectat (Nadeau-Bengio): fold-urile se suprapun, deci varianta e marita cu n_test / n_train
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * d.var(ddof=1))
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return t, p


def ttest_pair(folds, a, b, n_rows):
    k = len(folds[a])
    t, p = corrected_ttest(folds[a], folds[b], n_rows * (k - 1) / k, n_rows / k)
    verdict = "significant" if p < 0.05 else "NOT significant"
    print(f"{a} vs {b}: diff = {folds[a].mean() - folds[b].mean():+.3f}, t = {t:.2f}, p = {p:.3f} -> {verdict}")
    return p

## 1. Încărcarea datelor
🟦 **Ce face:** încearcă, în ordine: (1) copia locală `data/raw.csv`; (2) descărcarea de pe Kaggle cu `kagglehub`; (3) încărcarea manuală a fișierului în Colab. Salvează o copie în `data/raw.csv`.

🟩 **De ce `sep="\t"` la fișierul de pe Kaggle:** `marketing_campaign.csv` e de fapt separat prin **TAB**, nu prin virgulă — cu separatorul implicit pandas ar citi o singură coloană.

🟩 **De ce `kagglehub` și nu Google Drive:** setul e public, deci notebook-ul rulează la oricine, fără să monteze Drive-ul meu și fără cont Kaggle.

In [ ]:
RAW_CSV = f"{DATA_DIR}/raw.csv"
if os.path.exists(RAW_CSV):  # copia locala
    df_raw = pd.read_csv(RAW_CSV)
    print("Loaded from", RAW_CSV)
else:
    try:  # prima rulare: descarcam o singura data de pe Kaggle
        import kagglehub
        path = kagglehub.dataset_download("imakash3011/customer-personality-analysis")  # intoarce un FOLDER
        df_raw = pd.read_csv(f"{path}/marketing_campaign.csv", sep="\t")  # fisierul original e separat prin TAB
        print("Downloaded from Kaggle")
    except Exception as e:  # fara internet / Kaggle indisponibil -> incarcare manuala (raw.csv din ProiectA/data)
        print("Kaggle download failed:", e)
        from google.colab import files
        uploaded = files.upload()
        name = next(iter(uploaded))
        df_raw = pd.read_csv(name, sep="\t" if open(name).readline().count("\t") > 5 else ",")
    df_raw.to_csv(RAW_CSV, index=False)
    print("saved to", RAW_CSV)

print(df_raw.shape)
df_raw.head()

📊 **Ce a ieșit:** `(2240, 29)` — 2240 de clienți, 29 de coloane.

## 2. Analiza exploratorie a datelor (EDA)
Scopul EDA nu e să fac grafice „de decor”, ci să găsesc **probleme** în date și să iau **decizii** (tabelul de la finalul capitolului: observație → decizie).

### 2.1 Structura tabelului
🟦 **Ce face:** tipurile coloanelor, valorile lipsă, duplicatele (cu și fără `ID`), coloanele constante, coloanele de tip identificator, categoriile, valorile extreme suspecte.

🟩 **De ce verific duplicatele și fără `ID`:** `ID` e unic pe fiecare rând, deci *ascunde* duplicatele: același client introdus de două ori cu ID diferit nu apare ca duplicat. Fără `ID` apar **182** de rânduri duplicate. Dacă le-aș păstra, același client ar putea ajunge și în train, și în test → scor de test umflat.

In [ ]:
df_raw.info()

print("\nduplicated rows:", df_raw.duplicated().sum())
print("duplicated rows without ID:", df_raw.drop(columns=["ID"]).duplicated().sum())  # ID ascunde duplicatele
print("constant columns:", [c for c in df_raw.columns if df_raw[c].nunique(dropna=False) == 1])
print("ID-like columns:", [c for c in df_raw.columns if df_raw[c].nunique() == len(df_raw)])

for c in ["Education", "Marital_Status"]:
    print(f"\n{c}:", df_raw[c].value_counts().to_dict())

# valori imposibile / coduri ascunse
print("\nsmallest Year_Birth:", df_raw["Year_Birth"].nsmallest(4).tolist())
print("largest Income:", df_raw["Income"].nlargest(3).tolist())
print("Dt_Customer dtype:", df_raw["Dt_Customer"].dtype, "| example:", df_raw["Dt_Customer"].iloc[0])

📊 **Ce a ieșit:**
- `Income` are 24 de valori lipsă; restul coloanelor sunt complete.
- 0 duplicate cu `ID`, **182 fără `ID`**.
- `Z_CostContact` (=3) și `Z_Revenue` (=11) sunt **constante** → nu ajută la predicție (dar le refolosesc la profit, cap. 11.2).
- `Education` are „2n Cycle” (ciclul 2 Bologna = master), `Marital_Status` are „Absurd”, „YOLO”, „Alone”.
- `Year_Birth` = 1893, 1899, 1900 → imposibil (clienți de 115+ ani).
- `Income` maxim = **666666**, următorul = 162397 → nu e un client real, e un **cod de „lipsă”**.
- `Dt_Customer` e text (`dd-mm-yyyy`) → trebuie transformată în număr.

### 2.2 Statistici descriptive, valori lipsă, ținte
🟦 **Ce face:** pentru fiecare coloană numerică: % lipsă, medie, mediană, abatere, min, max, numărul de valori extreme după regula IQR și asimetria (skew). Apoi distribuția țintelor.

🟩 **De ce mă uit la medie vs. mediană și la skew:** dacă media ≫ mediana și skew > 1, distribuția are coadă lungă la dreapta → aleg mediana la completare și MAE la regresie (mai robuste).

🟧 **De ce nu elimin valorile extreme IQR:** la `Mnt*` sunt sute de „outlieri” IQR, dar sunt **clienți reali care cheltuie mult** — exact cei mai importanți pentru business. Regula IQR presupune o distribuție simetrică; aici nu e cazul. Elimin doar valorile **imposibile** (666666, anul 1893), nu pe cele mari.

In [ ]:
# copie doar pentru statistici: fara ID / coloane constante, plus tinta de regresie
df_stats = df_raw.drop(columns=["ID", "Z_CostContact", "Z_Revenue"]).assign(**{TARGET_REG: df_raw[MNT_COLS].sum(axis=1)})


def numeric_summary(data):
    rows = {}
    for c in data.select_dtypes(include="number").columns:
        s = data[c]
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows[c] = {"missing %": 100 * s.isna().mean(), "mean": s.mean(), "median": s.median(), "std": s.std(),
                   "min": s.min(), "max": s.max(),
                   "IQR outliers": ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum(), "skew": s.skew()}
    return pd.DataFrame(rows).T.round(2)


display(numeric_summary(df_stats))

missing = df_stats.isna().sum()
print("missing values:", missing[missing > 0].to_dict())

counts = df_stats[TARGET_CLF].value_counts()
print("\nclassification target:", df_stats[TARGET_CLF].value_counts(normalize=True).round(3).to_dict(),
      "| imbalance ratio:", round(counts.max() / counts.min(), 2))
print(f"regression target: mean {df_stats[TARGET_REG].mean():.0f}, median {df_stats[TARGET_REG].median():.0f}, "
      f"skew {df_stats[TARGET_REG].skew():.2f}")

📊 **Ce a ieșit:** `Income` skew 6,76; `Mnt*` skew ≈ 1–2 (medie ≈ 2× mediana). **Ținta de clasificare: 85,1% / 14,9%** → raport de dezechilibru 5,7. **Ținta de regresie:** medie 606, mediană 396, skew 0,86.

➡️ Consecință: acuratețea simplă ar fi înșelătoare (un model care spune mereu „0” are 85%) → am nevoie de metrici care văd clasa rară.

### 2.3 Legătura atributelor cu țintele
🟦 **Ce face:** mediana atributelor pe cele două clase, rata de răspuns pe categorii, corelația **Spearman** cu fiecare țintă.

🟩 **De ce mediana pe clase:** coloanele sunt asimetrice, media ar fi dominată de câțiva clienți.

🟩 **De ce `normalize="index"` la crosstab:** grupurile au mărimi foarte diferite (1127 Graduation vs. 54 Basic); compar **rate**, nu numere absolute.

🟩 **De ce Spearman și nu Pearson:** Spearman lucrează pe ranguri → prinde relații **monotone neliniare** și nu e influențat de valorile extreme. Pearson prinde doar relații liniare.

In [ ]:
# mediana pe clase (mediana: coloanele sunt asimetrice)
cols = ["Income", "Recency", "MntWines", "MntMeatProducts", "NumCatalogPurchases", "Kidhome", "Teenhome"]
display(df_stats.groupby(TARGET_CLF)[cols].median().T.rename(columns={0: "median, Response=0", 1: "median, Response=1"}))

# rata de raspuns pe categorii (normalize="index": grupurile au marimi foarte diferite)
for c in ["Education", "Marital_Status"]:
    display(pd.crosstab(df_stats[c], df_stats[TARGET_CLF], normalize="index").round(3).rename(columns={1: "response rate"})[["response rate"]].T)

spearman = df_stats.select_dtypes(include="number").corr(method="spearman")
print(f"correlation with {TARGET_CLF} (Spearman):")
print(spearman[TARGET_CLF].drop(TARGET_CLF).sort_values(key=abs, ascending=False).head(8).round(2).to_dict())
print(f"\ncorrelation with {TARGET_REG} (Spearman), without the Mnt* columns (it is their sum):")
print(spearman[TARGET_REG].drop([TARGET_REG] + MNT_COLS).sort_values(key=abs, ascending=False).head(8).round(2).to_dict())

📊 **Ce a ieșit:**
- Cei care răspund au venit mai mare (64k vs. 50k), au cumpărat **mai recent** (`Recency` 30 vs. 52 zile), cheltuie de ~3× mai mult pe vin și carne, nu au adolescenți acasă.
- Rata de răspuns crește cu educația: Basic 3,7% → PhD 20,8% (confirmă codificarea ordinală).
- Cele mai corelate cu `Response`: răspunsurile la campaniile anterioare (`AcceptedCmp5` 0,33, `AcceptedCmp1` 0,29) → cine a acceptat o ofertă înainte acceptă din nou.
- Cu `MntTotal`: `NumCatalogPurchases` 0,89, `Income` 0,85, `NumStorePurchases` 0,80 — **atenție**, `Num*Purchases` sunt suspecte de leakage (cap. 4 și 5.3).

### 2.4 Grafice (fiecare titlu e o întrebare)
🟦 **Ce face:** 6 grafice, fiecare răspunde la o întrebare: e ținta echilibrată? cum e distribuită cheltuiala? diferă răspunsul după educație? explică venitul cheltuiala? au cumpărat mai recent cei care răspund? ce atribute sunt corelate?

🟩 **De ce titluri-întrebare:** fiecare grafic trebuie să justifice o decizie; un titlu de tip „Histogramă Income” nu spune de ce e acolo.

🟩 **De ce scot `Income = 666666` doar din scatter:** altfel tot graficul s-ar comprima într-un colț; e doar pentru afișare, datele nu se modifică aici.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
labels = ["No (0)", "Yes (1)"]
colors = ["indianred", "seagreen"]
no_outlier = df_stats[df_stats["Income"] < 200_000]  # fara 666666, doar pentru afisare

# 1) Este tinta de clasificare echilibrata?
vc = df_stats[TARGET_CLF].value_counts().sort_index()
axes[0, 0].bar(labels, vc.values, color=colors)
for i, v in enumerate(vc.values):
    axes[0, 0].text(i, v, f"{100 * v / vc.sum():.1f}%", ha="center", va="bottom")
axes[0, 0].set(title=f"Is the target ({TARGET_CLF}) balanced?", ylabel="customers")

# 2) Cum e distribuita tinta de regresie?
axes[0, 1].hist(df_stats[TARGET_REG], bins=40, color="steelblue")
axes[0, 1].axvline(df_stats[TARGET_REG].mean(), color="k", linestyle="--", label="mean")
axes[0, 1].axvline(df_stats[TARGET_REG].median(), color="orange", linestyle="--", label="median")
axes[0, 1].set(title=f"How is total spending ({TARGET_REG}) distributed?", xlabel=TARGET_REG, ylabel="customers")
axes[0, 1].legend()

# 3) Difera rata de raspuns in functie de educatie?
pd.crosstab(df_stats["Education"], df_stats[TARGET_CLF], normalize="index").plot(
    kind="bar", stacked=True, ax=axes[0, 2], rot=0, color=colors)
axes[0, 2].set(title="Does the response rate differ by education?", ylabel="share of customers")
axes[0, 2].legend(labels, title=TARGET_CLF)

# 4) Explica venitul cat cheltuie un client?
for k, color, label in zip([0, 1], colors, labels):
    part = no_outlier[no_outlier[TARGET_CLF] == k]
    axes[1, 0].scatter(part["Income"], part[TARGET_REG], s=8, alpha=0.5, color=color, label=label)
axes[1, 0].set(title="Does income drive total spending?", xlabel="Income", ylabel=TARGET_REG)
axes[1, 0].legend(title=TARGET_CLF)

# 5) Au cumparat mai recent cei care au raspuns?
axes[1, 1].boxplot([df_stats.loc[df_stats[TARGET_CLF] == k, "Recency"] for k in [0, 1]], tick_labels=labels)
axes[1, 1].set(title="Did responders buy more recently?", xlabel=TARGET_CLF, ylabel="Recency (days)")

# 6) Ce atribute sunt corelate?
heat_cols = ["Income", "Kidhome", "Teenhome", "Recency", "NumDealsPurchases", "NumWebPurchases",
             "NumCatalogPurchases", "NumStorePurchases", "NumWebVisitsMonth", TARGET_REG, TARGET_CLF]
corr = df_stats[heat_cols].corr()
im = axes[1, 2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(len(heat_cols)), heat_cols, rotation=45, ha="right", fontsize=8)
axes[1, 2].set_yticks(range(len(heat_cols)), heat_cols, fontsize=8)
for i in range(len(heat_cols)):
    for j in range(len(heat_cols)):
        axes[1, 2].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6)
axes[1, 2].set_title("Which features are correlated? (Pearson)")
fig.colorbar(im, ax=axes[1, 2], shrink=0.8)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_01_explore.png", dpi=150, bbox_inches="tight")
plt.show()

### Ce am observat → ce am decis

| Observație | Decizie | De ce |
|---|---|---|
| `ID` unic, `Z_CostContact` = 3 și `Z_Revenue` = 11 constante | le elimin | nu aduc informație (Z_* le refolosesc la profit, cap. 11.2) |
| 0 duplicate cu `ID`, 182 fără `ID` | elimin duplicatele | același client nu trebuie să fie și în train, și în test |
| `Income = 666666` | → `NaN`, completat în pipeline | e un cod de lipsă; **contribuție**: păstrez clientul, nu șterg rândul |
| `Year_Birth` 1893, 1899, 1900 | → `NaN`, rândurile rămân | valori imposibile, restul rândului e valid |
| `Marital_Status`: Absurd, YOLO, Alone | Alone → Single; Absurd, YOLO → Other | „Alone” înseamnă același lucru ca „Single”; celelalte n-au sens |
| `Education = 2n Cycle` | unit cu Master (**contribuție**) | „2n Cycle” e ciclul 2 Bologna = master; apoi codificare ordinală |
| `Dt_Customer` e text | → `Customer_Days` (vechimea în zile față de 31.12.2014) | modelele au nevoie de numere |
| `Income`, `Mnt*` foarte asimetrice | mediana la completare, MAE la regresie | robuste la valori extreme |
| `Response` 85% / 15% | PR-AUC, ponderi pe clase | acuratețea ar fi înșelătoare |

## 3. Curățarea datelor
🟦 **Ce face:** aplică deciziile de mai sus, pas cu pas, și salvează `data/clean.csv`.

🟩 **De ce se poate face înainte de împărțirea train/test:** niciun pas de aici **nu învață** nimic din date (nu calculează medii, mediane, frecvențe). Sunt reguli fixe (`666666 → NaN`, `Alone → Single`), care dau același rezultat pe orice rând, indiferent ce alte rânduri există. De aceea nu produc leakage. Tot ce învață din date (completarea, scalarea, codificarea) e în pipeline.

🟩 **De ce `NaN` și nu ștergerea rândului:** clientul cu `Income` lipsă are restul de 25 de coloane valide; ștergerea ar pierde informație. Completarea se face în pipeline, cu mediana din train.

🟩 **De ce o dată de referință FIXĂ (31.12.2014) pentru `Customer_Days`:** dacă aș folosi „azi” (`pd.Timestamp.now()`), valorile s-ar schimba la fiecare rulare → rezultate nereproductibile. 31.12.2014 e după ultima dată de înscriere din set.

🟧 **De ce nu `Age = 2024 - Year_Birth`:** pentru modele, `Year_Birth` și vârsta sunt aceeași informație (diferă printr-o constantă), deci păstrez `Year_Birth`. Vârsta o construiesc doar în 9.1, ca atribut agregat.

In [ ]:
df = df_raw.copy()

# 1-2) identificatorul si coloanele constante nu aduc informatie
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])

# 3) duplicate ascunse de ID
df = df.drop_duplicates()

# 4-5) cod ascuns si valori imposibile -> NaN (randurile raman)
df.loc[df["Income"] == 666666, "Income"] = np.nan
df.loc[df["Year_Birth"] < 1920, "Year_Birth"] = np.nan

# 6-7) categorii cu acelasi inteles / fara sens
df["Marital_Status"] = df["Marital_Status"].replace({"Alone": "Single", "Absurd": "Other", "YOLO": "Other"})
df["Education"] = df["Education"].replace({"2n Cycle": "Master"})

# 8) data inscrierii -> vechimea clientului in zile, fata de o data FIXA
REFERENCE_DATE = pd.Timestamp("2014-12-31")
df["Customer_Days"] = (REFERENCE_DATE - pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")).dt.days
df = df.drop(columns=["Dt_Customer"])

# 9) tinta de regresie
df[TARGET_REG] = df[MNT_COLS].sum(axis=1)

df.to_csv(f"{DATA_DIR}/clean.csv", index=False)
print("before:", df_raw.shape, "-> after:", df.shape)
print("missing after cleaning (filled later, inside the pipeline):", df.isna().sum()[df.isna().sum() > 0].to_dict())
print("Education:", df["Education"].value_counts().to_dict())
print("Marital_Status:", df["Marital_Status"].value_counts().to_dict())

📊 **Ce a ieșit:** `(2240, 29) → (2058, 27)` (182 duplicate eliminate; −3 coloane inutile −`Dt_Customer` +`Customer_Days` +`MntTotal`). Rămân 25 `Income` și 3 `Year_Birth` lipsă — completate în pipeline.

## 4. Atributele, seturile de date și designul mulțimii de antrenament

🟦 **Ce face:** construiește `X` și `y` pentru fiecare problemă, împarte 80/20 în train/test și definește fold-urile de validare încrucișată.

**Ce scot din X și de ce:**

| Problema | Scot din X | De ce |
|---|---|---|
| clasificare | `MntTotal` | e suma exactă a coloanelor `Mnt*`, care rămân în X → redundantă |
| regresie | cele 6 `Mnt*` | ținta **e** suma lor → leakage direct (R² = 1 trivial) |
| regresie | `Num*Purchases` | numărate în **aceeași perioadă de 2 ani** ca ținta: le știm abia după ce clientul a cheltuit → leakage (verificat cu cifre în 5.3) |
| regresie | `Response` | rezultatul ultimei campanii nu e cunoscut pentru un client nou |

**Designul:**
- 🟩 **80/20 cu `random_state=SEED`:** 412 clienți la test — destul pentru o estimare finală, iar 1646 rămân pentru antrenare/validare.
- 🟩 **Stratificat la clasificare:** cu 15% pozitivi, o împărțire aleatoare poate da întâmplător 12% sau 18% în test; `stratify` păstrează proporția (15,2% train, 15,3% test).
- 🟩 **Împărțire aleatoare, nu în timp:** datele nu sunt o serie temporală (fiecare rând e un client, nu o observație în timp), deci nu am nevoie de split cronologic.
- 🟩 **Testul se folosește O SINGURĂ DATĂ, în capitolul 11.** Toate alegerile (modele, hiperparametri, prag) se fac prin validare încrucișată pe train. Dacă aș alege modelul după test, scorul de test n-ar mai fi o estimare nepărtinitoare.
- 🟩 **5 fold-uri amestecate, aceleași pentru toate modelele:** 5 e compromisul uzual între varianța estimării și timp; fold-uri identice → diferențele dintre modele nu vin din împărțiri diferite, iar testul t pe perechi e valid. `StratifiedKFold` la clasificare (≈ 50 de pozitivi în fiecare fold), `KFold` la regresie.

🟧 **De ce nu doar un set de validare fix (train/val/test):** cu 1646 de rânduri, un singur split de validare ar da un scor foarte zgomotos (± câteva procente doar din noroc); CV folosește fiecare rând o dată la validare și dă și abaterea.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold

LEAKY_R = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases", "NumDealsPurchases"]

# CLASIFICARE
Xc = df.drop(columns=[TARGET_CLF, TARGET_REG])
yc = (df[TARGET_CLF] == POS_CLASS).astype(int)

# REGRESIE
Xr = df.drop(columns=[TARGET_REG, TARGET_CLF] + MNT_COLS + LEAKY_R)
yr = df[TARGET_REG]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=SEED)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)

num_cols_c = Xc_train.select_dtypes(include="number").columns.tolist()
num_cols_r = Xr_train.select_dtypes(include="number").columns.tolist()

cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_r = KFold(n_splits=5, shuffle=True, random_state=SEED)

print("classification: train", Xc_train.shape, "test", Xc_test.shape,
      f"| class 1: train {yc_train.mean():.3f}, test {yc_test.mean():.3f}")
print("regression:     train", Xr_train.shape, "test", Xr_test.shape)
print("regression features:", Xr_train.columns.tolist())

📊 **Ce a ieșit:** clasificare train (1646, 25) / test (412, 25); regresie train (1646, 15) / test (412, 15). Proporția clasei 1: 0,152 vs. 0,153.

## 5. Preprocesarea în pipeline
Pipeline-ul are 3 drumuri (vezi `make_prep` din 0.2): numerice (mediană + scalare), `Education` (ordinal), `Marital_Status` (one-hot). Aici **verific alegerile cu cifre**, nu le iau „pe încredere”.

### 5.1 Valorile lipsă: merită un indicator de lipsă?
🟦 **Ce face:** compară rata de răspuns a clienților fără `Income` cu a celor cu `Income`.

🟩 **De ce:** dacă lipsa în sine e informativă (ex.: cine nu-și declară venitul răspunde mai rar), un indicator binar „Income lipsea” poate ajuta modelul. `SimpleImputer(add_indicator=True)` adaugă această coloană.

In [ ]:
no_income = Xc_train["Income"].isna()
print(f"rows without Income in train: {no_income.sum()} | response rate without Income: "
      f"{yc_train[no_income].mean():.3f} vs with Income: {yc_train[~no_income].mean():.3f}")
print("-> difference, but very few rows: add_indicator is tested in the grid search (8.1)")

📊 **Ce a ieșit:** doar 22 de rânduri fără `Income`; rata lor de răspuns e 4,5% vs. 15,3%. Diferența există, dar pe prea puține rânduri ca să decid „din ochi” → **las grid search-ul să decidă** (`add_indicator` e un hiperparametru în 8.1).

### 5.2 Scalarea: SVM pe atribute nescalate și SVR pe ținta nescalată
🟦 **Ce face:** același SVM/SVR, cu și fără scalare, pe aceleași fold-uri.

🟩 **De ce contează scalarea la SVM:** kernelul RBF calculează `exp(-γ·‖x − x'‖²)`, adică **distanțe**. `Income` are valori de ordinul 50 000, `Kidhome` 0–2 → fără scalare, distanța e dominată aproape doar de `Income` și restul atributelor sunt ignorate.

🟩 **De ce scalez și ținta la SVR:** `epsilon` (lățimea tubului în care erorile nu sunt penalizate, implicit 0,1) și `C` sunt în **unitățile țintei**. Pe ținta brută (sute–mii), ε = 0,1 e minuscul și C = 1 e o regularizare uriașă → modelul prezice aproape o constantă. `TransformedTargetRegressor` scalează ținta la antrenare și o readuce în unități monetare la predicție.

🟧 **De ce compar `StandardScaler` cu `RobustScaler`:** `RobustScaler` folosește mediana și IQR, deci e mai puțin influențat de valorile extreme. Dacă diferența e mică, păstrez `StandardScaler` (standardul); oricum îl testez în grid (8.1).

🟧 **De ce nu scalez la arbori:** arborii fac praguri pe câte un atribut („Income < 40 000?”) — scalarea nu schimbă nimic, deci `scaler="passthrough"`.

In [ ]:
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, LinearRegression

variants_c = {
    "SVC rbf, no scaling": make_pipe(SVC(class_weight="balanced"), num_cols_c, scaler="passthrough"),
    "SVC rbf + StandardScaler": make_pipe(SVC(class_weight="balanced"), num_cols_c),
    "SVC rbf + RobustScaler": make_pipe(SVC(class_weight="balanced"), num_cols_c, scaler=RobustScaler()),
}
table_scale_c, _ = compare(variants_c, Xc_train, yc_train, "classification", cv_c)
display(table_scale_c[["PR_AUC", "PR_AUC_std", "recall", "bal_acc"]].round(3))

variants_r = {
    "SVR rbf, no scaling": make_pipe(SVR(), num_cols_r, scaler="passthrough"),
    "SVR rbf + scaling X": make_pipe(SVR(), num_cols_r),
    "SVR rbf + scaling X and y": TransformedTargetRegressor(make_pipe(SVR(), num_cols_r), transformer=StandardScaler()),
}
table_scale_r, _ = compare(variants_r, Xr_train, yr_train, "regression", cv_r)
display(table_scale_r[["MAE", "MAE_std", "R2"]].round(3))

📊 **Ce a ieșit:**
- SVM: **fără scalare PR-AUC 0,307**, cu `StandardScaler` 0,540, cu `RobustScaler` 0,560 → scalarea aproape dublează scorul. Diferența Standard vs. Robust (0,02) e mult sub abatere (±0,08–0,10).
- SVR: scalare X și y → **R² 0,786**; doar X scalat → **R² 0,010** (practic o constantă!); nimic scalat → 0,237.

➡️ Concluzie: preprocesarea nu e un detaliu — schimbă complet rezultatul modelelor bazate pe distanțe.

### 5.3 Tratarea data leakage
**(a) Demonstrație:** selecția atributelor făcută înainte de validarea încrucișată, pe un set de **zgomot pur** (5000 de atribute aleatoare, țintă aleatoare). Răspunsul corect e „nu se poate prezice nimic” → acuratețe ≈ 0,5.

🟦 **Ce face:** varianta GREȘITĂ alege cele 20 de atribute „cele mai corelate” cu ținta pe **toate** rândurile, apoi face CV. Varianta CORECTĂ pune selecția **în** pipeline.

🟩 **De ce e greșită prima:** selecția a văzut deja etichetele fold-urilor de validare; dintre 5000 de atribute aleatoare, unele sunt corelate cu ținta din pură întâmplare, iar CV-ul „confirmă” această întâmplare.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(SEED)
X_noise = rng.normal(size=(200, 5000))
y_noise = rng.integers(0, 2, size=200)
cv_noise = StratifiedKFold(5, shuffle=True, random_state=SEED)

# GRESIT: selectia vede toate randurile (si fold-urile de validare)
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=cv_noise)
# CORECT: selectia e un pas din pipeline -> se invata doar pe partea de antrenare a fiecarui fold
right = cross_val_score(Pipeline([("select", SelectKBest(f_classif, k=20)), ("model", LogisticRegression(max_iter=1000))]),
                        X_noise, y_noise, cv=cv_noise)
print(f"selection BEFORE CV: accuracy {wrong.mean():.3f} (optimistic) | selection INSIDE the pipeline: {right.mean():.3f}")

📊 **Ce a ieșit:** selecție **înainte** de CV → acuratețe **0,825** pe zgomot (complet fals!); selecție **în** pipeline → 0,595 (aproape de 0,5, cât e corect). Asta e motivul pentru care toată preprocesarea mea stă în pipeline.

**(b) Contribuție: leakage-ul `Num*Purchases` la regresie.**

🟩 **Ipoteza:** `NumWebPurchases`, `NumCatalogPurchases`, `NumStorePurchases`, `NumDealsPurchases` sunt numărate în **aceeași fereastră de 2 ani** ca `MntTotal`. Pentru un client nou (pe care vreau să-l estimez) nu le cunosc încă — sunt practic un „rezumat” al cheltuielii. Cu ele, scorul ar fi umflat de o informație care nu există în momentul predicției.

🟦 **Ce face:** același model, pe aceleași rânduri și fold-uri, cu și fără aceste coloane.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

Xr_leaky = df.drop(columns=[TARGET_REG, TARGET_CLF] + MNT_COLS).loc[Xr_train.index]  # aceleasi randuri de train
leak_models = {
    "linear, WITH Num*Purchases": (make_pipe(LinearRegression(), Xr_leaky.select_dtypes("number").columns.tolist()), Xr_leaky),
    "linear, without (honest)": (make_pipe(LinearRegression(), num_cols_r), Xr_train),
    "HistGB, WITH Num*Purchases": (make_pipe(HistGradientBoostingRegressor(random_state=SEED),
                                              Xr_leaky.select_dtypes("number").columns.tolist(), scaler="passthrough"), Xr_leaky),
    "HistGB, without (honest)": (make_pipe(HistGradientBoostingRegressor(random_state=SEED), num_cols_r,
                                            scaler="passthrough"), Xr_train),
}
rows = []
for name, (model, X) in leak_models.items():
    row, _ = evaluate(model, X, yr_train, "regression", cv_r)
    rows.append({"model": name, **row})
display(pd.DataFrame(rows).set_index("model")[["MAE", "R2"]].round(3))

📊 **Ce a ieșit:** HistGB cu `Num*Purchases`: MAE 129, **R² 0,856**; fără: MAE 192, **R² 0,786**. La liniar: 0,805 vs. 0,715. Diferența e mare → coloanele chiar „trădează” ținta. Raportez varianta **onestă** (fără ele), chiar dacă scorul e mai mic.

🟧 **De ce nu le scot și la clasificare:** acolo ținta e răspunsul la **campania următoare**, iar istoricul de cumpărături e cunoscut înainte de campanie → nu e leakage.

## 6. Dezechilibrul claselor
Clasa 1 are ≈ 15% din train. Compar 5 strategii pe aceleași fold-uri, cu regresia logistică (model rapid și stabil, ca să văd efectul strategiei, nu al modelului).

| Strategie | Ce face |
|---|---|
| none | nimic — modelul optimizează eroarea totală, dominată de clasa 0 |
| `class_weight="balanced"` | erorile pe clasa rară costă de ~5,6× mai mult în funcția de pierdere |
| random oversampling | copiază aleator rânduri din clasa 1 până la egalitate |
| SMOTE | creează rânduri **sintetice** din clasa 1, interpolând între vecini |
| random undersampling | aruncă rânduri din clasa 0 până la egalitate |

🟩 **De ce reeșantionarea stă ÎN pipeline (`imblearn.Pipeline`):** trebuie aplicată **doar pe partea de antrenare** a fiecărui fold. Dacă aș face SMOTE înainte de CV, rânduri sintetice create din clienți de validare ar ajunge în antrenare → leakage, iar validarea ar conține date artificiale.

🟧 **De ce `imblearn.Pipeline` și nu cel din sklearn:** pipeline-ul din sklearn nu știe să aplice un pas care schimbă **numărul de rânduri** doar la `fit` și să-l sară la `predict`.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

print(f"class 1 in train: {yc_train.mean():.3f} | imbalance ratio: {(1 - yc_train.mean()) / yc_train.mean():.2f}")


def resampled(sampler):
    return ImbPipeline([("prep", make_prep(num_cols_c)), ("sampler", sampler),
                        ("model", LogisticRegression(max_iter=1000))])


strategies = {
    "none": make_pipe(LogisticRegression(max_iter=1000), num_cols_c),
    "class_weight=balanced": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced"), num_cols_c),
    "random oversampling": resampled(RandomOverSampler(random_state=SEED)),
    "SMOTE": resampled(SMOTE(random_state=SEED)),
    "random undersampling": resampled(RandomUnderSampler(random_state=SEED)),
}
table_imb, _ = compare(strategies, Xc_train, yc_train, "classification", cv_c)
display(table_imb[["recall", "precision", "F1", "bal_acc", "PR_AUC", "PR_AUC_std"]].round(3))

📊 **Ce a ieșit:**

| | recall | precizie | PR-AUC |
|---|---|---|---|
| none | 0,43 | 0,65 | 0,604 |
| class_weight | **0,79** | 0,44 | 0,601 |
| oversampling | 0,77 | 0,43 | 0,596 |
| SMOTE | 0,76 | 0,44 | 0,590 |
| undersampling | 0,77 | 0,41 | 0,572 |

**Decizia: `class_weight="balanced"`.**
- 🟩 recall-ul aproape se dublează (0,43 → 0,79): găsim ~80% din clienții care ar răspunde;
- 🟩 e la egalitate (sau mai bun) cu reeșantionarea, **fără rânduri artificiale** și fără timp suplimentar;
- 🟩 e disponibil direct în SVM, random forest, HistGradientBoosting și Keras → aceeași strategie pentru toate modelele.
- 🟧 **De ce nu SMOTE:** nu aduce nimic în plus aici, creează clienți care nu există și interpolarea pe atribute binare/ordinale (`AcceptedCmp*`, `Education`) dă valori fără sens (0,37 campanii acceptate).
- 🟧 **De ce nu undersampling:** aruncă ~80% din clasa 0 → cel mai mic PR-AUC.

**Observație importantă:** PR-AUC rămâne practic același — ponderile **nu schimbă ordinea** clienților după probabilitate, ci mută doar pragul efectiv. De aceea, mai târziu, aleg și pragul explicit (cap. 11).

## 7. Alegerea modelelor, antrenarea și evaluarea inițială (validare încrucișată)

### Metricile
**Clasificare — principala: PR-AUC** (aria sub curba precizie–recall, `average_precision`).
- 🟩 se concentrează pe **clasa rară** (precizia și recall-ul sunt despre clasa 1);
- 🟩 **nu depinde de prag** → compar modelele independent de pragul ales;
- 🟩 baseline-ul e ușor de interpretat: un model aleator are PR-AUC = proporția pozitivilor = **0,15**.
- 🟧 **De ce nu acuratețea:** „toți = 0” are 85% acuratețe și e inutil.
- 🟧 **De ce nu ROC-AUC ca principală:** la dezechilibru, ROC-AUC e optimist (rata de fals pozitivi are la numitor cei ~1400 de negativi, deci multe alarme false abia o mișcă). O raportez ca secundară.
- Secundare: ROC-AUC, F1, recall, precizie, acuratețe echilibrată.

**Regresie — principala: MAE** (eroarea absolută medie).
- 🟩 se interpretează direct: „greșim în medie cu X unități monetare”;
- 🟩 nu e dominată de câțiva clienți foarte mari (ținta e asimetrică); RMSE pătratizează erorile și le amplifică pe cele mari.
- Secundare: RMSE, R².

### Modelele (de la simplu la complex)
| Model | De ce l-am inclus |
|---|---|
| **baseline** (`DummyClassifier(prior)` / media) | pragul minim: orice model trebuie să-l bată clar |
| **regresie logistică / liniară** | model liniar, rapid, interpretabil — referința „simplă” |
| **kNN (k = 15)** | model bazat pe vecini; k = 15 ca să nu fie prea zgomotos |
| **arbore de decizie (adâncime 5)** | neliniar, interpretabil; adâncimea limitată împotriva overfitting-ului |
| **SVM / SVR (RBF)** | cerut de barem; kernelul RBF prinde frontiere neliniare |
| **random forest** | ansamblu de tip *bagging*: mulți arbori pe eșantioane bootstrap, medie → varianță mică |
| **HistGradientBoosting** | ansamblu de tip *boosting*: arbori adăugați secvențial, fiecare corectează erorile celor de dinainte; rapid, tratează nativ valorile lipsă |

🟩 **De ce `loss="absolute_error"` la HistGB regresie:** optimizează direct MAE, metrica mea principală.
🟩 **De ce `early_stopping=True` cu `max_iter=1000`:** modelul se oprește singur când validarea internă nu mai crește → nu trebuie să ghicesc numărul de arbori.
🟩 **De ce `min_samples_leaf=5` la random forest:** frunze cu un singur client înseamnă memorare; 5 netezește puțin.
🟧 **De ce `GradientBoosting` clasic doar în 8.4:** `HistGradientBoosting` e mult mai rapid (lucrează pe histograme) și dă rezultate echivalente; pe cel clasic îl folosesc doar pentru curbele pe arbori (`staged_predict`).

Toate primesc **aceeași preprocesare** și **aceleași fold-uri**; arborii fără scalare.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)

N_TREES = 300
models_c = {
    "baseline": make_pipe(DummyClassifier(strategy="prior"), num_cols_c),
    "logistic regression": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced"), num_cols_c),
    "kNN (k=15)": make_pipe(KNeighborsClassifier(n_neighbors=15), num_cols_c),
    "decision tree (depth 5)": make_pipe(DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=SEED),
                                         num_cols_c, scaler="passthrough"),
    "SVM (rbf)": make_pipe(SVC(class_weight="balanced"), num_cols_c),
    "random forest": make_pipe(RandomForestClassifier(N_TREES, min_samples_leaf=5, class_weight="balanced",
                                                      random_state=SEED, n_jobs=-1), num_cols_c, scaler="passthrough"),
    "HistGB": make_pipe(HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, early_stopping=True,
                                                       class_weight="balanced", random_state=SEED),
                        num_cols_c, scaler="passthrough"),
}
models_r = {
    "baseline (mean)": make_pipe(DummyRegressor(), num_cols_r),
    "linear regression": make_pipe(LinearRegression(), num_cols_r),
    "kNN (k=15)": make_pipe(KNeighborsRegressor(n_neighbors=15), num_cols_r),
    "decision tree (depth 5)": make_pipe(DecisionTreeRegressor(max_depth=5, random_state=SEED), num_cols_r,
                                         scaler="passthrough"),
    "SVR (rbf)": TransformedTargetRegressor(make_pipe(SVR(), num_cols_r), transformer=StandardScaler()),
    "random forest": make_pipe(RandomForestRegressor(N_TREES, min_samples_leaf=5, random_state=SEED, n_jobs=-1),
                               num_cols_r, scaler="passthrough"),
    "HistGB": make_pipe(HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.05, max_iter=1000,
                                                      early_stopping=True, random_state=SEED),
                        num_cols_r, scaler="passthrough"),
}
table_init_c, folds_init_c = compare(models_c, Xc_train, yc_train, "classification", cv_c)
table_init_r, folds_init_r = compare(models_r, Xr_train, yr_train, "regression", cv_r)
display(table_init_c.round(3))
display(table_init_r.round(3))

📊 **Ce a ieșit (CV pe 5 fold-uri):**
- **Clasificare:** regresia logistică **0,601** PR-AUC > random forest 0,566 > HistGB 0,553 > SVM 0,540 > kNN 0,462 > arbore 0,322 > baseline 0,152.
- **Regresie:** HistGB **MAE 185,5** (R² 0,795) > random forest 192 > SVR 194 > arbore 217 > liniară 233 > kNN 234 > baseline 527.

**Interpretare:**
- La clasificare, **modelul liniar câștigă** → frontiera dintre clase e aproape liniară (confirmat și de harta SVM din 8.2).
- Random forest are **0,988 pe train vs. 0,566 pe validare** → overfitting puternic (pe 1646 de rânduri, arborii adânci memorează).
- La regresie, modelele neliniare bat clar liniarul (185 vs. 233) → relația venit/copii → cheltuială e neliniară.

### 7.1 Curbele ROC și precizie–recall (predicții out-of-fold pe train)
🟦 **Ce face:** `cross_val_predict` prezice fiecare client din train cu un model antrenat pe celelalte 4 fold-uri → fiecare predicție e „onestă” (modelul nu a văzut clientul). Apoi desenez curbele ROC și PR pentru 5 modele.

🟩 **De ce out-of-fold și nu pe test:** testul e închis până la capitolul 11.
🟩 **De ce `decision_function` la SVC:** `SVC` fără `probability=True` nu are `predict_proba`; scorul de decizie dă **aceeași ordine** a clienților, iar curbele ROC/PR depind doar de ordine.
🟩 **Linia de referință:** la ROC diagonala (aleator), la PR o linie orizontală la 0,15 (proporția pozitivilor).

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay


def oof_scores(model, X, y, cv):
    # SVC nu are predict_proba -> folosim decision_function (aceeasi ordine a clientilor)
    method = "decision_function" if not hasattr(model, "predict_proba") else "predict_proba"
    s = cross_val_predict(model, X, y, cv=cv, method=method)
    return s[:, 1] if s.ndim == 2 else s


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in ["logistic regression", "SVM (rbf)", "random forest", "HistGB", "decision tree (depth 5)"]:
    s = oof_scores(models_c[name], Xc_train, yc_train, cv_c)
    RocCurveDisplay.from_predictions(yc_train, s, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(yc_train, s, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k:", label="random")
axes[1].axhline(yc_train.mean(), color="k", linestyle=":", label=f"baseline ({yc_train.mean():.2f})")
axes[0].set_title("ROC curve (out-of-fold, train)")
axes[1].set_title("Precision-recall curve (out-of-fold, train)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_02_roc_pr_cv.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Comparația vizuală: validare (medie ± abatere) vs. train
🟦 **Ce face:** bare = scorul pe validare cu abaterea pe fold-uri; liniuța neagră = scorul pe train. **Distanța dintre ele = overfitting.**

In [ ]:
# comparatia modelelor pe aceleasi fold-uri: media +- abaterea, validare vs train (overfitting)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, table, main in [(axes[0], table_init_c, "PR_AUC"), (axes[1], table_init_r, "MAE")]:
    t = table.iloc[::-1]
    ax.barh(t.index, t[main], xerr=t[f"{main}_std"], color="steelblue", label="validation (mean +- std)")
    ax.scatter(t[f"{main}_train"], t.index, color="k", marker="|", s=200, label="train")
    ax.set(title=f"{main} by model (5-fold CV)", xlabel=main)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_03_cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Selecția modelelor: grid search și analiza de senzitivitate

### 8.1 Regresia logistică: GridSearchCV + curba de validare pentru C
🟦 **Ce face:** caută cea mai bună combinație din: `C` ∈ {0,001 … 100} (6 valori), scalarea (Standard / Robust), indicatorul de lipsă (da / nu) → 24 de combinații × 5 fold-uri. Apoi curba de validare: scorul în funcție de `C`, cu ceilalți parametri fixați.

🟩 **De ce `C` pe scară logaritmică:** `C` = inversul puterii regularizării; efectul lui apare la schimbări de ordin de mărime (0,01 → 0,1 → 1), nu la 1 → 2.
🟩 **De ce caut și preprocesarea în grid:** pipeline-ul permite să tratez scalarea și `add_indicator` ca **hiperparametri** (`prep__num__scale`, `prep__num__impute__add_indicator`) → decizia din 5.1 o ia validarea, nu eu.
🟩 **De ce `GridSearchCV` (exhaustiv) aici:** spațiul e mic (24 de combinații) și modelul e rapid.
🟩 **De ce `return_train_score=True`:** ca să văd și overfitting-ul pentru fiecare combinație.

In [ ]:
from sklearn.model_selection import GridSearchCV, validation_curve

grid_lr = GridSearchCV(
    make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), num_cols_c),  # ponderile: decizia din cap. 6
    {"model__C": np.logspace(-3, 2, 6),
     "prep__num__scale": [StandardScaler(), RobustScaler()],
     "prep__num__impute__add_indicator": [False, True]},
    cv=cv_c, scoring="average_precision", n_jobs=-1, return_train_score=True)
grid_lr.fit(Xc_train, yc_train)  # 6 x 2 x 2 = 24 combinatii x 5 fold-uri
print("best:", grid_lr.best_params_, f"| CV PR-AUC = {grid_lr.best_score_:.3f}")
cols = ["param_model__C", "param_prep__num__scale",
        "param_prep__num__impute__add_indicator", "mean_test_score", "std_test_score", "mean_train_score"]
display(pd.DataFrame(grid_lr.cv_results_).sort_values("rank_test_score")[cols].head(5).round(3))

# senzitivitatea la C (ceilalti parametri fixati la cei mai buni)
C_range = np.logspace(-4, 3, 8)
train_s, val_s = validation_curve(grid_lr.best_estimator_, Xc_train, yc_train, param_name="model__C",
                                  param_range=C_range, cv=cv_c, scoring="average_precision", n_jobs=-1)
plt.figure(figsize=(7, 4))
plt.plot(C_range, train_s.mean(axis=1), "o-", label="train")
plt.errorbar(C_range, val_s.mean(axis=1), yerr=val_s.std(axis=1), fmt="o-", capsize=3, label="validation")
plt.xscale("log")
plt.gca().set(xlabel="C (log scale)", ylabel="PR-AUC", title="Logistic regression: how sensitive is PR-AUC to C?")
plt.legend()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_04_lr_C_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("validation PR-AUC by C:", dict(zip(C_range.round(4), val_s.mean(axis=1).round(3))))

📊 **Ce a ieșit:** cel mai bun: `C = 100`, `StandardScaler`, `add_indicator = True` → **PR-AUC 0,605**. Dar primele 5 combinații sunt toate 0,605 (de la C = 0,1 în sus) → **curba e plată**: modelul **nu e sensibil la C** peste 0,1. Doar la C foarte mic (0,0001 → 0,467) regularizarea prea puternică strică modelul (underfitting). Train 0,645 vs. validare 0,605 → overfitting mic, model stabil.

### 8.2 SVM (clasificare): grid pe kernel, C, γ + harta C × γ
🟦 **Ce face:** două sub-grile: kernel **liniar** (doar C) și kernel **RBF** (C × γ, 5 × 5). Total 29 de combinații × 5 fold-uri. Apoi harta de căldură C × γ pe train și pe validare.

🟩 **De ce o listă de două grile:** `gamma` nu are sens la kernelul liniar; o grilă unică ar repeta degeaba combinațiile liniare pentru fiecare γ.
🟩 **Ce înseamnă C și γ:** **C** = cât de tare penalizez punctele greșit clasificate (C mare → margine îngustă, risc de overfitting); **γ** = cât de „local” e kernelul (γ mare → fiecare punct influențează doar vecinii foarte apropiați → frontieră foarte sinuoasă, overfitting).
🟩 **De ce harta pe train ȘI pe validare:** pe train scorul crește mereu cu C și γ; pe validare se vede unde începe overfitting-ul.

In [ ]:
param_grid_svc = [
    {"model__kernel": ["linear"], "model__C": np.logspace(-2, 1, 4)},
    {"model__kernel": ["rbf"], "model__C": np.logspace(-1, 3, 5), "model__gamma": np.logspace(-4, 0, 5)},
]
grid_svc = GridSearchCV(make_pipe(SVC(class_weight="balanced"), num_cols_c), param_grid_svc, cv=cv_c,
                        scoring="average_precision", n_jobs=-1, return_train_score=True)
t0 = time.time()
grid_svc.fit(Xc_train, yc_train)  # 4 + 25 = 29 combinatii x 5 fold-uri
print(f"SVC search: {time.time() - t0:.0f} s | best: {grid_svc.best_params_} | CV PR-AUC = {grid_svc.best_score_:.3f}")


def heatmaps_C_gamma(grid, prefix="model__", fixed=None, title="", error=False):
    # harta C x gamma pentru kernelul rbf, pe train si pe validare
    res = pd.DataFrame(grid.cv_results_)
    res = res[res[f"param_{prefix}kernel"] == "rbf"]
    for key, value in (fixed or {}).items():
        res = res[res[f"param_{key}"] == value]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, split in zip(axes, ["train", "test"]):
        table = res.pivot_table(index=f"param_{prefix}C", columns=f"param_{prefix}gamma", values=f"mean_{split}_score")
        if error:
            table = -table
        im = ax.imshow(table.values, cmap="viridis_r" if error else "viridis", aspect="auto")
        ax.set_xticks(range(table.shape[1]), [f"{g:g}" for g in table.columns])
        ax.set_yticks(range(table.shape[0]), [f"{c:g}" for c in table.index])
        for i in range(table.shape[0]):
            for j in range(table.shape[1]):
                ax.text(j, i, f"{table.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
        ax.set(xlabel="gamma", ylabel="C", title=f"{title} - {'validation' if split == 'test' else 'train'}")
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    return fig


heatmaps_C_gamma(grid_svc, title="SVC rbf, PR-AUC")
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_05_svc_C_gamma.png", dpi=150, bbox_inches="tight")
plt.show()

📊 **Ce a ieșit:** cel mai bun: RBF, `C = 100`, `γ = 0,001` → **PR-AUC 0,606** (de la 0,540 cu setările implicite). Zona bună e o **diagonală** în hartă (C mare cu γ mic ≈ C mic cu γ mai mare — compromis între penalizare și netezime). Pentru γ ≥ 0,1, validarea scade la 0,28–0,44 deși train-ul crește → overfitting clar. γ = 0,001 înseamnă un kernel foarte „lat” → frontiera câștigătoare e **aproape liniară**, la fel ca regresia logistică.

### 8.3 SVR (regresie): grid pe kernel, C, γ, ε
🟦 **Ce face:** la fel ca la SVM, plus **ε** (lățimea tubului în care erorile nu sunt penalizate). Prefixul `regressor__model__` apare pentru că modelul e învelit în `TransformedTargetRegressor`. 60 de combinații × 5 fold-uri.

🟩 **De ce ε ∈ {0,05; 0,1; 0,3}:** ținta e scalată (abatere 1), deci ε = 0,1 înseamnă „ignor erorile sub 0,1 abateri standard” ≈ 60 de unități monetare.

In [ ]:
prefix = "regressor__model__"
param_grid_svr = [
    {f"{prefix}kernel": ["linear"], f"{prefix}C": np.logspace(-2, 1, 4), f"{prefix}epsilon": [0.05, 0.1, 0.3]},
    {f"{prefix}kernel": ["rbf"], f"{prefix}C": np.logspace(-1, 2, 4), f"{prefix}gamma": np.logspace(-3, 0, 4),
     f"{prefix}epsilon": [0.05, 0.1, 0.3]},
]
grid_svr = GridSearchCV(TransformedTargetRegressor(make_pipe(SVR(), num_cols_r), transformer=StandardScaler()),
                        param_grid_svr, cv=cv_r, scoring="neg_mean_absolute_error", n_jobs=-1, return_train_score=True)
t0 = time.time()
grid_svr.fit(Xr_train, yr_train)  # 12 + 48 = 60 combinatii x 5 fold-uri
print(f"SVR search: {time.time() - t0:.0f} s | best: {grid_svr.best_params_} | CV MAE = {-grid_svr.best_score_:.1f}")

heatmaps_C_gamma(grid_svr, prefix=prefix, error=True, title="SVR rbf, MAE",
                 fixed={f"{prefix}epsilon": grid_svr.best_params_.get(f"{prefix}epsilon")})
plt.show()

# senzitivitatea la epsilon (C, gamma fixati la cei mai buni)
res = pd.DataFrame(grid_svr.cv_results_)
best = grid_svr.best_params_
mask = (res[f"param_{prefix}kernel"] == best[f"{prefix}kernel"]) & (res[f"param_{prefix}C"] == best[f"{prefix}C"])
if best[f"{prefix}kernel"] == "rbf":
    mask &= res[f"param_{prefix}gamma"] == best[f"{prefix}gamma"]
print("CV MAE by epsilon:", dict(zip(res.loc[mask, f"param_{prefix}epsilon"], (-res.loc[mask, "mean_test_score"]).round(1))))

📊 **Ce a ieșit:** cel mai bun: RBF, `C = 100`, `γ = 0,01`, `ε = 0,05` → MAE 199,1. ε mai mic e mai bun (0,05: 199 → 0,3: 210) — un tub îngust forțează modelul să urmărească mai fidel ținta. Notă: SVR-ul reglat (199) iese puțin mai slab decât cel implicit din cap. 7 (194), pentru că combinația implicită (`C = 1`, `γ = "scale"`) nu se află în grilă → o grilă e doar o eșantionare a spațiului, nu garantează că găsește cel mai bun punct. HistGB rămâne mai bun.

### 8.4 HistGradientBoosting: RandomizedSearchCV + senzitivitatea la rata de învățare
🟦 **Ce face:** 30 de combinații **aleatoare** din distribuții pentru: rata de învățare, numărul maxim de frunze, numărul minim de exemple într-o frunză, regularizarea L2.

🟩 **De ce Randomized și nu Grid aici:** 4 hiperparametri continui/întregi; o grilă cu 5 valori fiecare = 625 de combinații. Căutarea aleatoare explorează mai bine spațiul cu un buget fix (30), mai ales când doar 1–2 parametri contează cu adevărat (Bergstra & Bengio, 2012).
🟩 **De ce `loguniform` pentru learning_rate și L2:** valorile utile se întind pe mai multe ordine de mărime (0,01 – 0,3); o distribuție uniformă ar alege aproape mereu valori mari.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

space = {
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_leaf_nodes": randint(8, 65),
    "model__min_samples_leaf": randint(10, 81),
    "model__l2_regularization": loguniform(1e-3, 10),
}
searches = {}
for task, base, X, y, cv, scoring in [
        ("classification", models_c["HistGB"], Xc_train, yc_train, cv_c, "average_precision"),
        ("regression", models_r["HistGB"], Xr_train, yr_train, cv_r, "neg_mean_absolute_error")]:
    search = RandomizedSearchCV(base, space, n_iter=30, cv=cv, scoring=scoring, random_state=SEED, n_jobs=-1,
                                return_train_score=True).fit(X, y)
    searches[task] = search
    sign = -1 if scoring.startswith("neg_") else 1
    table = table_init_c if task == "classification" else table_init_r
    default = table.loc["HistGB"].iloc[0]
    best_std = search.cv_results_["std_test_score"][search.best_index_]
    print(f"{task}: default {default:.3f} -> tuned {sign * search.best_score_:.3f} (std {best_std:.3f})")
    print("   best:", {k.replace("model__", ""): round(v, 4) for k, v in search.best_params_.items()})

📊 **Ce a ieșit:** clasificare: implicit 0,553 → reglat **0,603**; regresie: 185,5 → **182,5** MAE. Reglarea ajută la clasificare (`min_samples_leaf = 71` → arbori mult mai „conservatori” → mai puțin overfitting), dar foarte puțin la regresie (testul t din cap. 9 arată că −3 MAE nu e semnificativ).

**Senzitivitatea la rata de învățare:** antrenez `GradientBoosting` clasic cu 600 de arbori și măsor pierderea pe validare **după fiecare arbore** (`staged_predict`), pentru 3 rate de învățare.

🟩 **De ce `GradientBoosting` clasic aici:** are `staged_predict`, care dă predicția după 1, 2, …, 600 de arbori dintr-o singură antrenare.
🟩 **De ce un split de validare 80/20 din train:** curbele cer o singură validare, nu CV; testul rămâne neatins.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import log_loss, mean_absolute_error

# pierderea pe validare dupa fiecare arbore, pentru 3 rate de invatare (validare = 20% din train)
Xc_f, Xc_v, yc_f, yc_v = train_test_split(Xc_train, yc_train, test_size=0.2, stratify=yc_train, random_state=SEED)
Xr_f, Xr_v, yr_f, yr_v = train_test_split(Xr_train, yr_train, test_size=0.2, random_state=SEED)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for lr in [0.3, 0.1, 0.03]:
    gb_c = make_pipe(GradientBoostingClassifier(learning_rate=lr, n_estimators=600, max_depth=3, random_state=SEED),
                     num_cols_c, scaler="passthrough").fit(Xc_f, yc_f)
    loss_c = [log_loss(yc_v, p[:, 1]) for p in gb_c["model"].staged_predict_proba(gb_c["prep"].transform(Xc_v))]
    gb_r = make_pipe(GradientBoostingRegressor(loss="absolute_error", learning_rate=lr, n_estimators=600, max_depth=3,
                                               random_state=SEED), num_cols_r, scaler="passthrough").fit(Xr_f, yr_f)
    loss_r = [mean_absolute_error(yr_v, p) for p in gb_r["model"].staged_predict(gb_r["prep"].transform(Xr_v))]
    for ax, loss in [(axes[0], loss_c), (axes[1], loss_r)]:
        best = int(np.argmin(loss))
        ax.plot(range(1, len(loss) + 1), loss, label=f"lr = {lr}: min {loss[best]:.3f} at {best + 1} trees")
axes[0].set(title="Classification: validation log-loss", xlabel="number of trees", ylabel="log-loss")
axes[1].set(title="Regression: validation MAE", xlabel="number of trees", ylabel="MAE")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_06_boosting_lr.png", dpi=150, bbox_inches="tight")
plt.show()

📊 **Interpretare:** cu rata **0,3** minimul vine repede (puțini arbori), apoi validarea **crește** → overfitting. Cu **0,03** modelul învață încet și are nevoie de sute de arbori. ➡️ Rata de învățare și numărul de arbori sunt legate: **rată mai mică ⇒ mai mulți arbori**. De aceea folosesc `early_stopping`, care alege automat numărul de arbori.

### 8.5 Cât de optimist e scorul celei mai bune combinații? (nested CV)
🟦 **Ce face:** **bucla interioară** (3 fold-uri) face grid search-ul; **bucla exterioară** (aceleași 5 fold-uri) evaluează modelul ales pe date pe care căutarea **nu le-a văzut**.

🟩 **De ce:** `best_score_` din grid search e **optimist** — am ales maximul dintre 24 de scoruri zgomotoase, deci am ales și „norocul”. Nested CV estimează cât de bine generalizează **întreaga procedură** (căutare + antrenare).

In [ ]:
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
inner_grid = GridSearchCV(make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), num_cols_c),
                          {"model__C": np.logspace(-3, 2, 6), "prep__num__impute__add_indicator": [False, True]},
                          cv=inner, scoring="average_precision", n_jobs=-1)
nested = cross_val_score(inner_grid, Xc_train, yc_train, cv=cv_c, scoring="average_precision")
print(f"logistic regression: grid best_score_ {grid_lr.best_score_:.3f} | nested CV {nested.mean():.3f} +- {nested.std():.3f}")

📊 **Ce a ieșit:** `best_score_` 0,605 vs. **nested 0,601 ± 0,066** → optimismul e de doar 0,004. Spațiul căutat e mic și curba plată, deci alegerea hiperparametrilor nu „a supra-învățat” validarea. **0,60 e estimarea realistă** a performanței.

## 9. Ansambluri din modele diferite și comparația finală
🟦 **Ce face:** combină modele din **familii diferite** (liniar + SVM + arbori):
- **Voting (soft)** — media probabilităților celor 3 modele;
- **Stacking** — un meta-model (regresie logistică / `RidgeCV`) învață **cum** să combine predicțiile out-of-fold ale celor 3.

Apoi tabelul complet cu toate modelele reglate și testul t corectat.

🟩 **De ce familii diferite:** un ansamblu ajută doar dacă membrii greșesc **diferit**; 3 modele foarte asemănătoare greșesc la aceiași clienți.
🟩 **De ce `CalibratedClassifierCV` la SVM:** voting-ul soft are nevoie de probabilități; SVC dă doar un scor de decizie, iar calibrarea sigmoid (Platt) îl transformă în probabilitate.
🟩 **De ce `cv=5` la stacking:** meta-modelul trebuie antrenat pe predicții **out-of-fold**, altfel ar învăța să aibă încredere în membrii care au overfitting pe train.

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier, VotingRegressor, StackingRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import RidgeCV

best_lr = grid_lr.best_estimator_
best_svc = grid_svc.best_estimator_
best_svr = grid_svr.best_estimator_
hgb_c = searches["classification"].best_estimator_
hgb_r_default = models_r["HistGB"]

members_c = [("logistic", best_lr),
             ("svm", CalibratedClassifierCV(best_svc, method="sigmoid", cv=3)),  # SVC nu are predict_proba -> calibrat
             ("histgb", hgb_c)]
members_r = [("linear", models_r["linear regression"]), ("svr", best_svr), ("rf", models_r["random forest"])]

final_models_c = {
    "baseline": models_c["baseline"],
    "logistic regression (tuned)": best_lr,
    "SVM (tuned)": best_svc,
    "random forest": models_c["random forest"],
    "HistGB (tuned)": hgb_c,
    "voting (soft)": VotingClassifier(members_c, voting="soft", n_jobs=-1),
    "stacking": StackingClassifier(members_c, final_estimator=LogisticRegression(class_weight="balanced"), cv=5, n_jobs=-1),
}
final_models_r = {
    "baseline (mean)": models_r["baseline (mean)"],
    "linear regression": models_r["linear regression"],
    "SVR (tuned)": best_svr,
    "random forest": models_r["random forest"],
    "HistGB (default)": hgb_r_default,
    "HistGB (tuned)": searches["regression"].best_estimator_,
    "voting": VotingRegressor(members_r, n_jobs=-1),
    "stacking": StackingRegressor(members_r, final_estimator=RidgeCV(), cv=5, n_jobs=-1),
}
table_c, folds_c = compare(final_models_c, Xc_train, yc_train, "classification", cv_c)
table_r, folds_r = compare(final_models_r, Xr_train, yr_train, "regression", cv_r)
display(table_c.round(3))
display(table_r.round(3))
pd.concat([table_c.assign(task="classification"), table_r.assign(task="regression")]).to_csv(
    f"{RESULTS_DIR}/{MY_NAME}_cv_results.csv")

📊 **Ce a ieșit:**
- **Clasificare:** voting **0,645**, stacking 0,642, SVM reglat 0,606, logistică reglată 0,605, HistGB reglat 0,603, random forest 0,566.
- **Regresie:** HistGB reglat 182,5, HistGB implicit 185,5, stacking 186,1, random forest 192, voting 193, SVR 199, liniară 233.

Ansamblurile par mai bune la clasificare — dar sunt diferențele **reale** sau zgomot? Răspunde testul t.

### Testul t corectat (Nadeau–Bengio) între modelele de top
🟦 **Ce face:** pentru două modele, ia diferențele de scor pe cele **5 fold-uri comune** și testează dacă media diferențelor e diferită de 0, cu varianța corectată.

🟩 **Interpretare:** p < 0,05 → diferența e mai mare decât zgomotul fold-urilor; altfel modelele sunt la **egalitate statistică** și aleg pe criterii practice (simplitate, viteză, interpretabilitate).

In [ ]:
# sunt diferentele dintre modelele de top mai mari decat zgomotul fold-urilor?
ttest_pair(folds_c, table_c.index[0], table_c.index[1], len(Xc_train))
ttest_pair(folds_c, table_c.index[0], "logistic regression (tuned)", len(Xc_train))
ttest_pair(folds_c, "SVM (tuned)", "logistic regression (tuned)", len(Xc_train))
print()
ttest_pair(folds_r, table_r.index[0], table_r.index[1], len(Xr_train))
ttest_pair(folds_r, "HistGB (default)", "random forest", len(Xr_train))
_ = ttest_pair(folds_r, "SVR (tuned)", "linear regression", len(Xr_train))

📊 **Ce a ieșit:**
- voting vs. logistică: +0,039, **p = 0,14 → NU e semnificativ**; SVM vs. logistică: +0,000, p = 0,995.
- HistGB reglat vs. implicit: −3 MAE, p = 0,11 → NU e semnificativ.
- SVR vs. liniară: −34 MAE, **p = 0,004 → semnificativ** (relația e cu adevărat neliniară).

➡️ **Decizia (motivată):**
- Clasificare → **regresia logistică reglată**: la egalitate statistică cu voting/stacking, dar de ~15× mai rapidă, cu un singur model în loc de trei și **interpretabilă** (coeficienții = efectul fiecărui atribut).
- Regresie → **HistGB cu setările implicite**: reglarea a adus −3 MAE nesemnificativ; stacking-ul e la egalitate, dar de ~20× mai lent. Prefer modelul mai simplu (**principiul parcimoniei / Occam**).

### 9.1 Contribuție: atribute agregate pentru SVM-ul RBF
🟩 **Ipoteza:** kernelul RBF măsoară distanțe în spațiul tuturor atributelor; informația împrăștiată în mai multe coloane (5 coloane `AcceptedCmp*`, `Kidhome` + `Teenhome`) cântărește „de mai multe ori” în distanță. Dacă le adun în atribute agregate, distanța devine mai relevantă:
- `AcceptedTotal` = numărul de campanii acceptate (0–5);
- `Children` = numărul total de copii;
- `Age` = 2014 − `Year_Birth`;
- `SpendPerPurchase` = cheltuiala totală / numărul de cumpărături (cât cheltuie pe o vizită).

🟩 **De ce `FunctionTransformer` în pipeline fără risc de leakage:** funcția face doar operații **pe rândul curent** (sume, împărțiri), nu învață nimic din alte rânduri.
🟩 **De ce `clip(lower=1)`:** evită împărțirea la 0 pentru clienții fără cumpărături.
🟩 **Comparația e corectă:** același model (aceiași C, γ), aceleași fold-uri, testul t corectat.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone

ACC_COLS = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]
PURCHASE_COLS = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]


def add_features(X):
    X = X.copy()
    X["AcceptedTotal"] = X[ACC_COLS].sum(axis=1)
    X["Children"] = X["Kidhome"] + X["Teenhome"]
    X["Age"] = 2014 - X["Year_Birth"]
    X["SpendPerPurchase"] = X[MNT_COLS].sum(axis=1) / X[PURCHASE_COLS].sum(axis=1).clip(lower=1)
    return X.drop(columns=ACC_COLS + ["Kidhome", "Teenhome", "Year_Birth"])


new_num = add_features(Xc_train.head()).select_dtypes("number").columns.tolist()
svc_new = Pipeline([("features", FunctionTransformer(add_features)),  # nu invata nimic din date -> fara leakage
                    ("svm", make_pipe(clone(best_svc["model"]), new_num))])

row_old, f_old = evaluate(best_svc, Xc_train, yc_train, "classification", cv_c)
row_new, f_new = evaluate(svc_new, Xc_train, yc_train, "classification", cv_c)
display(pd.DataFrame([row_old, row_new], index=["SVM, original features", "SVM, + aggregated features"])
        [["PR_AUC", "PR_AUC_std", "ROC_AUC", "F1", "recall"]].round(3))
t, p = corrected_ttest(f_new, f_old, len(Xc_train) * 4 / 5, len(Xc_train) / 5)
print(f"diff = {f_new.mean() - f_old.mean():+.3f}, t = {t:.2f}, p = {p:.3f}")

📊 **Ce a ieșit:** PR-AUC 0,606 → **0,625** (+0,020), p = 0,09 → **ipoteză susținută parțial**: tendința e pozitivă, dar nu ajunge la pragul de 0,05 (fold-urile au doar ~50 de pozitivi, deci zgomotul e mare).

## 10. Rețele neuronale (Keras)
Rețele dense (straturi complet conectate), activare **ReLU**, optimizator **Adam**, ieșire **sigmoid** la clasificare (probabilitatea clasei 1) și **liniară** la regresie.

### 10.1 Datele pentru Keras și prima comparație (overfitting vs. regularizare)
🟦 **Ce face:** folosește același split 80/20 din **train** ca în 8.4 (testul rămâne închis): preprocesarea și scalarea țintei se învață doar pe partea „fit”. Apoi compară 4 variante:
1. rețea **mare** (256-256-128, ~100 000 de parametri) fără regularizare, 200 de epoci;
2. mare + **EarlyStopping**;
3. mare + **Dropout 0,3** + EarlyStopping;
4. rețea **mică** (32-16) + Dropout + EarlyStopping.

🟩 **De ce ReLU:** nu se saturează pentru valori pozitive (spre deosebire de sigmoid/tanh) → gradienții nu dispar, antrenare rapidă.
🟩 **De ce Adam:** își adaptează rata de învățare pentru fiecare pondere; funcționează bine cu valorile implicite (lr = 0,001).
🟩 **De ce EarlyStopping(patience=20, restore_best_weights=True):** oprește antrenarea când pierderea pe validare nu mai scade 20 de epoci și **revine la cele mai bune ponderi** → alege automat numărul de epoci.
🟩 **De ce Dropout:** la fiecare pas „stinge” aleator 30% din neuroni → rețeaua nu se poate baza pe neuroni anume, învață reprezentări mai robuste (efect de regularizare, similar unui ansamblu).
🟩 **De ce `class_weight` la clasificare:** aceeași decizie ca în cap. 6.
🟩 **De ce pierderea MAE la regresie:** optimizează direct metrica principală; MSE ar fi dominată de clienții foarte mari.
🟩 **De ce scalez ținta:** rețeaua pornește cu ponderi mici; o țintă de ordinul miilor ar da gradienți uriași la început.

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import keras
from keras import layers, regularizers, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import average_precision_score

# 20% din TRAIN = validare pentru retele (testul ramane inchis); aceleasi parti ca in 8.4
prep_net_c = make_prep(num_cols_c).fit(Xc_f)
Ac_f, Ac_v = (prep_net_c.transform(d).astype("float32") for d in (Xc_f, Xc_v))
tc_f, tc_v = yc_f.to_numpy().astype("float32"), yc_v.to_numpy().astype("float32")

prep_net_r = make_prep(num_cols_r).fit(Xr_f)
Ar_f, Ar_v = (prep_net_r.transform(d).astype("float32") for d in (Xr_f, Xr_v))
y_scaler = StandardScaler().fit(yr_f.to_numpy().reshape(-1, 1))  # tinta scalata, invatata doar pe partea "fit"
tr_f, tr_v = (y_scaler.transform(d.to_numpy().reshape(-1, 1)).ravel().astype("float32") for d in (yr_f, yr_v))


def to_money(t_scaled):
    return y_scaler.inverse_transform(np.asarray(t_scaled).reshape(-1, 1)).ravel()


def balanced_weights(y):
    w = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.asarray(y).astype(int))
    return {0: w[0], 1: w[1]}


def make_net(n_inputs, task, units=(32, 16), dropout=0.0, l2=0.0, lr=1e-3, loss=None):
    reg = regularizers.l2(l2) if l2 > 0 else None
    net = keras.Sequential([keras.Input(shape=(n_inputs,))])
    for u in units:
        net.add(layers.Dense(u, activation="relu", kernel_regularizer=reg))
        if dropout > 0:
            net.add(layers.Dropout(dropout))
    if task == "classification":
        net.add(layers.Dense(1, activation="sigmoid"))  # P(clasa 1)
        net.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss or "binary_crossentropy")
    else:
        net.add(layers.Dense(1))  # iesire liniara
        net.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss or "mse")
    return net


def early_stop():
    return callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)


data_net = {"classification": (Ac_f, tc_f, Ac_v, tc_v), "regression": (Ar_f, tr_f, Ar_v, tr_v)}


def val_score(task, p, t_v):
    # PR-AUC (clasificare) / MAE in unitati monetare (regresie), pe validare
    return average_precision_score(t_v, p) if task == "classification" else mean_absolute_error(yr_v, to_money(p))


BIG = (256, 256, 128)
variants = {"big, no regularization": dict(units=BIG, stop=False),
            "big + EarlyStopping": dict(units=BIG, stop=True),
            "big + Dropout 0.3 + ES": dict(units=BIG, dropout=0.3, stop=True),
            "small (32, 16) + Dropout + ES": dict(units=(32, 16), dropout=0.3, stop=True)}
rows, histories = [], {}
for task, (A_f, t_f, A_v, t_v) in data_net.items():
    for name, cfg in variants.items():
        cfg = dict(cfg)
        stop = cfg.pop("stop")
        keras.utils.set_random_seed(SEED)
        net = make_net(A_f.shape[1], task, loss="mae" if task == "regression" else None, **cfg)
        h = net.fit(A_f, t_f, validation_data=(A_v, t_v), epochs=200, batch_size=32, verbose=0,
                    callbacks=[early_stop()] if stop else [],
                    class_weight=balanced_weights(t_f) if task == "classification" else None)
        histories[(task, name)] = h
        rows.append({"task": task, "variant": name, "epochs run": len(h.history["loss"]), "params": net.count_params(),
                     "validation PR-AUC / MAE": val_score(task, net.predict(A_v, verbose=0).ravel(), t_v)})
display(pd.DataFrame(rows).set_index(["task", "variant"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, task in zip(axes, data_net):
    for name in variants:
        ax.plot(histories[(task, name)].history["val_loss"], label=name)
    ax.set(xlabel="epoch", ylabel="validation loss", title=f"{task}: validation loss", yscale="log")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_07_nets_regularization.png", dpi=150, bbox_inches="tight")
plt.show()

📊 **Ce a ieșit:**
- Clasificare: mare fără regularizare **0,479** → + EarlyStopping 0,522 → + Dropout **0,597**; mică 0,560.
- Regresie: mare fără regularizare MAE **229** → + ES 197 → + Dropout **187**; mică 187.
- Graficul: fără regularizare, pierderea pe validare **crește** după primele epoci (overfitting clasic: ~100 000 de parametri pentru ~1300 de rânduri).

➡️ Regularizarea (EarlyStopping + Dropout) e **esențială** pe un set atât de mic.

### 10.2 Validarea încrucișată a rețelei (aceleași fold-uri) + o grilă mică de hiperparametri
🟦 **Ce face:** `net_cv` reproduce validarea încrucișată „de mână” pentru Keras: în **fiecare fold** construiește o preprocesare nouă și o rețea nouă, învățate doar pe partea de antrenare; 15% din partea de antrenare e validare pentru EarlyStopping. Grila: arhitectură {(16), (32, 16)} × dropout {0; 0,3}, pe 3 fold-uri.

🟩 **De ce o funcție proprie și nu `cross_validate`:** rețeaua Keras nu e un estimator scikit-learn; scriu bucla explicit ca să garantez **aceleași reguli anti-leakage** ca la celelalte modele.
🟩 **De ce `clear_session()` și `set_random_seed` în fiecare fold:** fiecare fold pornește de la o rețea curată, reproductibilă, fără memorie de la fold-ul anterior.
🟩 **De ce o grilă mică și doar 3 fold-uri:** e cel mai lent pas (o rețea nouă pentru fiecare fold și configurație); 4 configurații × 3 fold-uri e un compromis rezonabil.

In [ ]:
from itertools import product
from sklearn.metrics import f1_score, recall_score, r2_score
from sklearn.neural_network import MLPClassifier, MLPRegressor


def net_cv(X, y, task, cv, num_cols, units=(32, 16), dropout=0.3, lr=1e-3, epochs=200, seed=SEED):
    # in fiecare fold: preprocesare + retea NOUA, invatate doar pe partea de antrenare a fold-ului
    scores = []
    y = np.asarray(y)
    for idx_fit, idx_val in cv.split(X, y):
        X_f, X_v, y_f, y_v = X.iloc[idx_fit], X.iloc[idx_val], y[idx_fit], y[idx_val]
        X_f, X_es, y_f, y_es = train_test_split(X_f, y_f, test_size=0.15, random_state=seed,
                                                stratify=y_f if task == "classification" else None)
        prep = make_prep(num_cols).fit(X_f)
        A_f, A_es, A_v = (prep.transform(d).astype("float32") for d in (X_f, X_es, X_v))
        if task == "regression":
            ys = StandardScaler().fit(y_f.reshape(-1, 1))
            t_f, t_es = (ys.transform(d.reshape(-1, 1)).ravel() for d in (y_f, y_es))
        else:
            t_f, t_es = y_f, y_es
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        net = make_net(A_f.shape[1], task, units=units, dropout=dropout, lr=lr,
                       loss="mae" if task == "regression" else None)
        net.fit(A_f, t_f, validation_data=(A_es, t_es), epochs=epochs, batch_size=32, verbose=0,
                callbacks=[early_stop()], class_weight=balanced_weights(t_f) if task == "classification" else None)
        p = net.predict(A_v, verbose=0).ravel()
        if task == "regression":
            scores.append(mean_absolute_error(y_v, ys.inverse_transform(p.reshape(-1, 1)).ravel()))
        else:
            scores.append(average_precision_score(y_v, p))
    return np.array(scores)


# grila mica: 2 x 2 = 4 configuratii, pe 3 fold-uri (o retea noua / fold / configuratie -> cel mai lent pas)
cv3_c = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv3_r = KFold(n_splits=3, shuffle=True, random_state=SEED)
grid_net = {"units": [(16,), (32, 16)], "dropout": [0.0, 0.3], "lr": [1e-3]}
best_cfg = {}
for task, X, y, cv, num_cols in [("classification", Xc_train, yc_train, cv3_c, num_cols_c),
                                 ("regression", Xr_train, yr_train, cv3_r, num_cols_r)]:
    rows = []
    t0 = time.time()
    for units, dropout, lr in product(*grid_net.values()):
        s = net_cv(X, y, task, cv, num_cols, units=units, dropout=dropout, lr=lr, epochs=150)
        rows.append({"units": units, "dropout": dropout, "lr": lr, "score": s.mean(), "std": s.std()})
    res = pd.DataFrame(rows).sort_values("score", ascending=(task == "regression")).reset_index(drop=True)
    best_cfg[task] = res.iloc[0]
    print(f"{task}: {len(rows)} configurations x 3 folds in {time.time() - t0:.0f} s (score = {'PR-AUC' if task == 'classification' else 'MAE'})")
    display(res.round(3))

📊 **Ce a ieșit:** clasificare: cea mai bună **(16) + dropout 0,3** (PR-AUC 0,561); regresie: **(32, 16) + dropout 0,3** (MAE 198). Dropout-ul câștigă în ambele cazuri — confirmă 10.1.

🟦 **Ce face:** rețeaua aleasă, pe **aceleași 5 fold-uri** ca restul modelelor, comparată cu `MLPClassifier`/`MLPRegressor` din scikit-learn și cu cel mai bun model clasic, plus testul t corectat.

In [ ]:
# reteaua aleasa, pe ACELEASI 5 fold-uri ca restul modelelor, vs. cel mai bun model si vs. MLP din scikit-learn
t0 = time.time()
cfg = best_cfg["classification"]
folds_net_c = net_cv(Xc_train, yc_train, "classification", cv_c, num_cols_c,
                     units=cfg["units"], dropout=float(cfg["dropout"]), lr=float(cfg["lr"]))
cfg = best_cfg["regression"]
folds_net_r = net_cv(Xr_train, yr_train, "regression", cv_r, num_cols_r,
                     units=cfg["units"], dropout=float(cfg["dropout"]), lr=float(cfg["lr"]))
print(f"network CV (2 x 5 folds): {time.time() - t0:.0f} s")

mlp_c = make_pipe(MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, early_stopping=True, max_iter=500,
                                random_state=SEED), num_cols_c)
mlp_r = TransformedTargetRegressor(make_pipe(MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-3, early_stopping=True,
                                                          max_iter=500, random_state=SEED), num_cols_r),
                                   transformer=StandardScaler())
_, folds_mlp_c = evaluate(mlp_c, Xc_train, yc_train, "classification", cv_c)
_, folds_mlp_r = evaluate(mlp_r, Xr_train, yr_train, "regression", cv_r)

folds_c |= {"Keras network": folds_net_c, "MLPClassifier": folds_mlp_c}
folds_r |= {"Keras network": folds_net_r, "MLPRegressor": folds_mlp_r}
table_net = pd.DataFrame([
    {"task": "classification", "model": m, "PR-AUC / MAE": folds_c[m].mean(), "std": folds_c[m].std()}
    for m in ["Keras network", "MLPClassifier", "logistic regression (tuned)", table_c.index[0]]] + [
    {"task": "regression", "model": m, "PR-AUC / MAE": folds_r[m].mean(), "std": folds_r[m].std()}
    for m in ["Keras network", "MLPRegressor", "HistGB (default)", table_r.index[0]]]).drop_duplicates(["task", "model"])
display(table_net.set_index(["task", "model"]).round(3))
ttest_pair(folds_c, "Keras network", "logistic regression (tuned)", len(Xc_train))
_ = ttest_pair(folds_r, "Keras network", "HistGB (default)", len(Xr_train))

📊 **Ce a ieșit:** clasificare: Keras **0,591** vs. logistică 0,605 (p = 0,60 → egalitate); regresie: Keras MAE **194** vs. HistGB 185,5 (p = 0,07). MLP din scikit-learn e mai slab (0,528 / 208) — nu are Dropout și `class_weight`.

➡️ Rețelele **nu bat** modelele clasice: pe date **tabelare mici** (1646 de rânduri, 25 de coloane), modelele liniare și ansamblurile de arbori sunt de obicei la fel de bune sau mai bune; rețelele au nevoie de mult mai multe date.

### 10.3 Contribuție: media a 5 rețele cu seed-uri diferite
🟩 **Ipoteza:** o rețea depinde de inițializarea aleatoare a ponderilor (seed). Media predicțiilor a 5 rețele (aceeași configurație, seed-uri 0–4) reduce această varianță → mai bună și mai stabilă decât o singură rețea. Tabelul arată și cât de mult variază o singură rețea între seed-uri.

In [ ]:
rows = []
for task, (A_f, t_f, A_v, t_v) in data_net.items():
    cfg = best_cfg[task]
    preds = []
    for seed in range(5):
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        net = make_net(A_f.shape[1], task, units=cfg["units"], dropout=float(cfg["dropout"]), lr=float(cfg["lr"]),
                       loss="mae" if task == "regression" else None)
        net.fit(A_f, t_f, validation_data=(A_v, t_v), epochs=200, batch_size=32, verbose=0, callbacks=[early_stop()],
                class_weight=balanced_weights(t_f) if task == "classification" else None)
        preds.append(net.predict(A_v, verbose=0).ravel())
    single = [val_score(task, p, t_v) for p in preds]
    rows.append({"task": task, "single net: mean of 5 seeds": np.mean(single), "single net: std": np.std(single),
                 "single net: min": np.min(single), "single net: max": np.max(single),
                 "ensemble of 5 nets": val_score(task, np.mean(preds, axis=0), t_v)})
display(pd.DataFrame(rows).set_index("task").round(3))

📊 **Ce a ieșit:**
- Regresie: o rețea 188,9 ± 2,8 (între 183 și 191) → media a 5: **184,0** — mai bună decât media unei singure rețele. ✅
- Clasificare: o rețea 0,602 ± 0,017 → media a 5: **0,611** — câștig mic.

➡️ Ipoteza e susținută mai ales la regresie. Arată și de ce raportez media pe seed-uri: o singură rețea poate varia cu ±0,02 doar din noroc.

## 11. Evaluarea finală pe setul de test
Modelele finale (motivarea în cap. 9 și 12): **regresia logistică reglată** la clasificare și **HistGradientBoosting** cu setările implicite + oprire timpurie la regresie.

🟦 **Ce face:**
1. **Alege pragul de decizie ÎNAINTE de test**, pe predicțiile out-of-fold din train: precizia maximă cu condiția **recall ≥ 0,75**.
2. Antrenează fiecare model pe **tot** train-ul.
3. Evaluează **o singură dată** pe test.
4. Calculează un **interval de încredere bootstrap 95%** pentru metrica principală.
5. Calculează **lift@20%**.

🟩 **De ce pragul ≠ 0,5:** cu `class_weight="balanced"`, probabilitățile sunt deplasate; iar 0,5 nu are nicio legătură cu obiectivul de business. Am cerut recall ≥ 0,75 (vreau să găsesc 3 din 4 clienți care ar răspunde — o ofertă ratată costă mai mult decât un contact în plus), apoi cea mai mare precizie.
🟩 **De ce pragul se alege pe train (out-of-fold):** dacă l-aș alege pe test, testul nu ar mai fi o evaluare independentă.
🟩 **De ce bootstrap:** testul are doar 63 de pozitivi → scorul de test are o incertitudine mare. Reeșantionez rândurile testului de 2000 de ori și iau percentilele 2,5% și 97,5%.
🟩 **De ce lift@20%:** e metrica pe care o înțelege marketingul: „dacă sun doar primii 20% clienți după model, câți din cei care răspund prind?”.

In [ ]:
from sklearn.metrics import (precision_score, roc_auc_score, root_mean_squared_error, classification_report,
                             ConfusionMatrixDisplay)

FINAL_C = "logistic regression (tuned)"
FINAL_R = "HistGB (default)"
final_c = clone(final_models_c[FINAL_C])
final_r = clone(final_models_r[FINAL_R])

# pragul, ales doar pe train (out-of-fold)
p_oof = cross_val_predict(final_c, Xc_train, yc_train, cv=cv_c, method="predict_proba")[:, 1]
thresholds = np.arange(0.05, 0.96, 0.01)
ok = [t for t in thresholds if recall_score(yc_train, p_oof >= t) >= 0.75]
THRESHOLD = max(ok, key=lambda t: precision_score(yc_train, p_oof >= t))
print(f"threshold chosen on train (OOF): {THRESHOLD:.2f}")

# antrenare pe tot train-ul, O SINGURA evaluare pe test
final_c.fit(Xc_train, yc_train)
final_r.fit(Xr_train, yr_train)
p_te = final_c.predict_proba(Xc_test)[:, 1]
pred_c = (p_te >= THRESHOLD).astype(int)
pred_r = final_r.predict(Xr_test)

test_c = {"PR_AUC": average_precision_score(yc_test, p_te), "ROC_AUC": roc_auc_score(yc_test, p_te),
          "F1": f1_score(yc_test, pred_c), "recall": recall_score(yc_test, pred_c),
          "precision": precision_score(yc_test, pred_c)}
test_r = {"MAE": mean_absolute_error(yr_test, pred_r), "RMSE": root_mean_squared_error(yr_test, pred_r),
          "R2": r2_score(yr_test, pred_r)}

# interval bootstrap 95% pentru metrica principala (reesantionam randurile testului)
rng = np.random.default_rng(SEED)


def bootstrap_ci(metric, y_true, y_score, n=2000):
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    idx = rng.integers(0, len(y_true), size=(n, len(y_true)))
    return np.percentile([metric(y_true[i], y_score[i]) for i in idx], [2.5, 97.5])


ci_c = bootstrap_ci(average_precision_score, yc_test, p_te)
ci_r = bootstrap_ci(mean_absolute_error, yr_test, pred_r)
print(f"\nCLASSIFICATION ({FINAL_C}): CV PR-AUC {folds_c[FINAL_C].mean():.3f} | TEST PR-AUC {test_c['PR_AUC']:.3f} "
      f"[95% CI {ci_c[0]:.3f} - {ci_c[1]:.3f}]")
print({k: round(v, 3) for k, v in test_c.items()})
print(classification_report(yc_test, pred_c, digits=3))
print(f"REGRESSION ({FINAL_R}): CV MAE {folds_r[FINAL_R].mean():.1f} | TEST MAE {test_r['MAE']:.1f} "
      f"[95% CI {ci_r[0]:.1f} - {ci_r[1]:.1f}]")
print({k: round(v, 3) for k, v in test_r.items()})

# lift: ce parte din cei care raspund se afla in primii 20% clienti ordonati de model
top20 = np.argsort(-p_te)[: int(0.2 * len(p_te))]
captured = yc_test.to_numpy()[top20].sum() / yc_test.sum()
print(f"top 20% of customers contain {captured:.0%} of all responders -> lift@20% = {captured / 0.2:.2f}")

pd.DataFrame([{"task": "classification", "model": FINAL_C, "threshold": THRESHOLD, **test_c, "CI_low": ci_c[0], "CI_high": ci_c[1]},
              {"task": "regression", "model": FINAL_R, **test_r, "CI_low": ci_r[0], "CI_high": ci_r[1]}]
             ).to_csv(f"{RESULTS_DIR}/{MY_NAME}_test.csv", index=False)
joblib.dump(final_c, f"{MODELS_DIR}/{MY_NAME}_final_clf.joblib")
joblib.dump(final_r, f"{MODELS_DIR}/{MY_NAME}_final_reg.joblib")

📊 **Ce a ieșit:**
- Prag ales pe train: **0,54**.
- **Clasificare (test, 412 clienți, 63 pozitivi):** PR-AUC **0,744** [IC 95%: 0,633 – 0,856], ROC-AUC 0,917, **recall 0,825**, precizie 0,525 (vs. 15% fără model → de ~3,5× mai precis decât contactarea la întâmplare), F1 0,642.
- **Lift@20% = 3,89:** primii 20% clienți conțin **78%** din cei care răspund.
- **Regresie (test):** MAE **160,0** [144,7 – 176,0], RMSE 230, **R² 0,826**.

**De ce testul e mai bun decât CV-ul (0,744 vs. 0,605)?** Testul are doar 63 de pozitivi → variabilitate mare (vezi IC-ul larg, care include și valori ~0,63). Plus, modelul final e antrenat pe tot train-ul (cu 25% mai multe date decât în fiecare fold). **Nu am schimbat nimic după ce am văzut testul**; estimarea realistă rămâne cea din CV / nested CV (≈ 0,60).

### 11.1 Graficele pe test
🟦 **Ce face:** curba ROC și PR pe test cu punctul pragului ales (roșu), matricea de confuzie; la regresie: predicție vs. real, reziduuri și eroarea pe sferturi (quartile) ale țintei.

🟩 **De ce eroarea pe quartile:** MAE-ul global ascunde **unde** greșește modelul; tabelul arată dacă eroarea e uniformă sau concentrată la clienții mari.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
RocCurveDisplay.from_predictions(yc_test, p_te, name=FINAL_C, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k:")
fpr_t = ((p_te >= THRESHOLD) & (yc_test == 0)).sum() / (yc_test == 0).sum()
axes[0].scatter(fpr_t, test_c["recall"], color="red", zorder=3, label=f"threshold {THRESHOLD:.2f}")
axes[0].set_title("Test: ROC curve")
PrecisionRecallDisplay.from_predictions(yc_test, p_te, name=FINAL_C, ax=axes[1])
axes[1].scatter(test_c["recall"], test_c["precision"], color="red", zorder=3, label=f"threshold {THRESHOLD:.2f}")
axes[1].axhline(yc_test.mean(), color="k", linestyle=":")
axes[1].set_title("Test: precision-recall curve")
for ax in axes[:2]:
    ax.legend(fontsize=8)
ConfusionMatrixDisplay.from_predictions(yc_test, pred_c, ax=axes[2], colorbar=False)
axes[2].set_title(f"Test: confusion matrix (threshold {THRESHOLD:.2f})")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_08_test_classification.png", dpi=150, bbox_inches="tight")
plt.show()

resid = yr_test - pred_r
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(yr_test, pred_r, s=10, alpha=0.5)
axes[0].plot([0, yr_test.max()], [0, yr_test.max()], "k--", linewidth=1)
axes[0].set(xlabel=f"real {TARGET_REG}", ylabel="predicted", title="Test: predicted vs. real")
axes[1].scatter(pred_r, resid, s=10, alpha=0.5)
axes[1].axhline(0, color="k", linestyle="--", linewidth=1)
axes[1].set(xlabel="predicted", ylabel="residual = real - predicted", title="Test: residuals")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_09_test_regression.png", dpi=150, bbox_inches="tight")
plt.show()

quart = pd.qcut(yr_test, 4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"])
display(pd.DataFrame({"MAE": resid.abs(), "mean residual": resid}).groupby(quart, observed=True).mean().round(1))

📊 **Ce a ieșit (regresie):** MAE crește cu cheltuiala (Q1: 38 → Q4: 269). Reziduul mediu e **negativ** la Q1–Q3 (supraestimare) și **pozitiv** la Q4 (+143, subestimare) → modelul „trage spre medie”: e prudent cu clienții foarte mari, normal pentru un model care minimizează eroarea absolută cu atribute limitate (fără istoric de cumpărături).

### 11.2 Ce atribute folosesc modelele? (importanța prin permutare, pe test)
🟦 **Ce face:** amestecă aleator valorile unei coloane (de 20 de ori) și măsoară cât **scade** scorul. Dacă scade mult, modelul depinde de acel atribut.

🟩 **De ce permutare și nu `feature_importances_`/coeficienți:** funcționează **la fel** pentru orice model (logistică și HistGB), măsoară importanța în termenii **metricii mele** (PR-AUC / MAE), pe date nevăzute, și lucrează pe coloanele **originale** (nu pe cele one-hot). Importanța din arbori (Gini) favorizează atributele cu multe valori distincte.

In [ ]:
from sklearn.inspection import permutation_importance

imp_c = permutation_importance(final_c, Xc_test, yc_test, scoring="average_precision", n_repeats=20, random_state=SEED, n_jobs=-1)
imp_r = permutation_importance(final_r, Xr_test, yr_test, scoring="neg_mean_absolute_error", n_repeats=20, random_state=SEED, n_jobs=-1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, imp, X, title in [(axes[0], imp_c, Xc_test, "Classification: drop in PR-AUC"),
                          (axes[1], imp_r, Xr_test, "Regression: increase in MAE")]:
    top = pd.Series(imp.importances_mean, index=X.columns).sort_values().tail(10)
    ax.barh(top.index, top.values, color="steelblue")
    ax.set(title=f"Permutation importance on test - {title}")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_10_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### 11.3 Contribuție: pragul ales după profit
Coloanele constante eliminate la curățare sunt chiar **costul și câștigul campaniei**: un contact costă `Z_CostContact` = 3, un client care acceptă aduce `Z_Revenue` = 11.

`profit = 11 × (clienți contactați care acceptă) − 3 × (clienți contactați)`

🟩 **Ipoteza:** pragul care maximizează profitul diferă de cel ales după F1 sau după recall.
🟩 **Corectitudinea:** pragurile se aleg **doar pe train** (out-of-fold); coloana „test” doar raportează.

In [ ]:
REVENUE, COST = 11, 3


def profit(y_true, contacted):
    y_true, contacted = np.asarray(y_true).astype(bool), np.asarray(contacted).astype(bool)
    return REVENUE * (y_true & contacted).sum() - COST * contacted.sum()


profits = [profit(yc_train, p_oof >= t) for t in thresholds]
T_PROFIT = thresholds[int(np.argmax(profits))]
T_F1 = thresholds[int(np.argmax([f1_score(yc_train, p_oof >= t) for t in thresholds]))]

rows = []
for name, c_tr, c_te in [("contact everyone", np.ones(len(yc_train)), np.ones(len(yc_test))),
                         (f"threshold max F1 ({T_F1:.2f})", p_oof >= T_F1, p_te >= T_F1),
                         (f"threshold recall>=0.75 ({THRESHOLD:.2f})", p_oof >= THRESHOLD, p_te >= THRESHOLD),
                         (f"threshold max profit ({T_PROFIT:.2f})", p_oof >= T_PROFIT, p_te >= T_PROFIT)]:
    rows.append({"strategy": name, "contacted (train)": np.mean(c_tr),
                 "profit per customer (train, OOF)": profit(yc_train, c_tr) / len(yc_train),
                 "profit per customer (test)": profit(yc_test, c_te) / len(yc_test)})
display(pd.DataFrame(rows).set_index("strategy").round(3))

plt.figure(figsize=(7, 3.8))
plt.plot(thresholds, np.array(profits) / len(yc_train))
plt.axvline(T_PROFIT, color="k", linestyle="--", linewidth=1, label=f"max profit ({T_PROFIT:.2f})")
plt.axhline(0, color="gray", linewidth=0.8)
plt.gca().set(xlabel="threshold", ylabel="profit per customer (train, OOF)", title="Which threshold earns the most?")
plt.legend()
plt.savefig(f"{FIGURES_DIR}/{MY_NAME}_11_profit_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

📊 **Ce a ieșit:**
- **Contactez pe toată lumea:** profit **−1,33 pe client** (pierdere: doar 15% răspund, 0,15 × 11 < 3).
- **Cu modelul (prag 0,53):** contactez doar ~26% din clienți → **+0,52 pe client (train) / +0,68 (test)**.
- Pragul optim după profit (0,53) **coincide** cu cel după F1 și e foarte aproape de cel ales de mine (0,54) → ipoteza **nu se confirmă**, dar e o veste bună: alegerea mea de prag e și cea mai profitabilă.

## 12. Concluzii

**1. Ce clienți vor accepta oferta? — Da, parțial se poate prezice.** Am comparat regresia logistică, kNN, arborele de decizie, SVM (liniar și RBF), random forest, HistGradientBoosting, voting, stacking și rețele neuronale. Am ales **regresia logistică cu ponderi pe clase**: bate clar baseline-ul (PR-AUC 0,605 față de 0,152 în CV) și e la egalitate statistică cu ansamblurile (voting +0,039, p = 0,14; SVM +0,000, p = 0,995), fiind de ≈ 15 ori mai rapidă și explicabilă. Hiperparametrii reglați: C (curba e plată de la C = 0,1 în sus), scalarea și indicatorul de lipsă pentru `Income`. Ponderile pe clase ridică recall-ul de la 0,43 la 0,79. Pragul 0,54 a fost ales pe train. **Pe test** (412 clienți, 63 pozitivi): PR-AUC 0,744 [0,633 – 0,856], ROC-AUC 0,917, recall 0,825, precizie 0,525 (față de 15% fără model), lift@20% = 3,89.

**2. Cât cheltuie un client în 2 ani? — Da, destul de bine.** Am ales **HistGradientBoosting** cu setările implicite și oprire timpurie: MAE 185,5 în CV, **160,0 pe test [144,7 – 176,0], R² 0,826**. Reglarea a adus doar −3 MAE (p = 0,11), iar stacking-ul e la egalitate, dar de ≈ 20 de ori mai lent. Relația e neliniară: SVR-ul RBF bate semnificativ regresia liniară (−34 MAE, p = 0,004).

**Preprocesarea a contat:** fără scalare, SVM-ul are PR-AUC 0,31 în loc de 0,54–0,56, iar SVR-ul fără ținta scalată are R² 0,01. Fără eliminarea leakage-ului, regresia ar fi raportat R² 0,86 în loc de 0,79, cu o informație pe care nu o avem în momentul predicției.

**Comportamentul la hiperparametri:** modelele flexibile fac ușor overfitting pe 1646 de rânduri (random forest are 0,99 pe train). La SVM, zona bună e o diagonală în harta C × γ, iar pentru γ ≥ 0,1 validarea scade la 0,28–0,44: frontiera câștigătoare e aproape liniară. La boosting, o rată de învățare mai mică cere mai mulți arbori.

**Gradul de încredere:** abaterile între fold-uri sunt mari la clasificare (±0,05–0,07 PR-AUC), pentru că un fold are ≈ 50 de clienți pozitivi. De aceea diferențele dintre modelele de top nu sunt semnificative. Testul iese mai bun decât CV-ul (0,744 față de 0,605), dar nu am schimbat nimic după test. Estimarea realistă e cea din CV / nested CV (0,601 ± 0,066).

**Contribuții originale:**
- 666666 → `NaN` (cod de lipsă, nu client real) și 2n Cycle = Master.
- Leakage-ul `Num*Purchases` la regresie (R² 0,86 → 0,79).
- Atribute agregate pentru SVM: +0,020 PR-AUC, p = 0,09, ipoteză susținută parțial.
- Pragul după profit: contactarea tuturor pierde 1,33 pe client, modelul câștigă +0,52 (train) / +0,68 (test). Pragul optim după profit (0,53) coincide cu cel după F1.
- Media a 5 rețele: la regresie e mai bună decât o rețea medie (184,0 față de 188,9 ± 2,8); la clasificare câștigul e mic (0,611 față de 0,602 ± 0,017). Rețelele nu bat modelele clasice.

**Limitări:** set mic (63 de pozitivi la test → IC larg); datele sunt dintr-o singură perioadă (nu știm dacă modelul rămâne valabil în timp); la regresie modelul subestimează clienții foarte mari.

---
## Anexă: întrebări posibile la prezentare (și răspunsuri scurte)

**De ce nu ai folosit acuratețea?** Clasele sunt 85/15; un model care spune mereu „nu” are 85% acuratețe și nu găsește niciun client. PR-AUC pornește de la 0,15 pentru un model inutil și arată cât de bine ordonăm clienții din clasa rară.

**Ce e data leakage și cum l-ai evitat?** Informație din afara datelor de antrenare care ajunge în model (din validare/test sau din viitor). L-am evitat: (1) toată preprocesarea care învață din date e în `Pipeline`, re-antrenată pe fiecare fold; (2) reeșantionarea e în `imblearn.Pipeline`; (3) am scos `Mnt*` și `Num*Purchases` de la regresie; (4) testul l-am folosit o singură dată; (5) pragul l-am ales pe out-of-fold din train. Demonstrația din 5.3(a): pe zgomot, selecția greșită dă 0,825 în loc de ~0,5.

**De ce curățarea e înainte de split, dar completarea în pipeline?** Curățarea folosește reguli fixe (nu calculează nimic din date). Completarea cu mediana calculează mediana → trebuie învățată doar din train.

**De ce `class_weight` și nu SMOTE?** Rezultate la egalitate, fără date sintetice, fără timp în plus, disponibil la toate modelele. SMOTE interpolează și atribute binare (valori fără sens).

**De ce regresia logistică și nu voting-ul, care are scorul mai mare?** Diferența (+0,039) nu e semnificativă statistic (p = 0,14, test t corectat). La egalitate aleg modelul mai simplu, mai rapid și interpretabil.

**De ce testul t corectat?** Fold-urile de antrenare se suprapun → scorurile nu sunt independente → testul t clasic dă p-valori prea mici. Corecția Nadeau–Bengio mărește varianța cu `n_test/n_train`.

**Ce e nested CV și de ce?** CV în CV: interiorul alege hiperparametrii, exteriorul evaluează alegerea pe date nevăzute de căutare. Arată cât de optimist e `best_score_` (aici doar 0,004).

**De ce ai scalat ținta la SVR?** `epsilon` și `C` sunt în unitățile țintei; pe ținta brută, SVR prezice aproape o constantă (R² 0,01).

**De ce rețelele nu câștigă?** Set tabelar mic (1646 de rânduri); rețelele au mulți parametri și fac overfitting fără regularizare puternică; modelele liniare și de arbori sunt de obicei la fel de bune pe astfel de date.

**De ce testul e mai bun decât CV-ul?** Doar 63 de pozitivi la test → variabilitate mare (IC 0,63 – 0,86); modelul final are și mai multe date de antrenare. Nu am ajustat nimic după test.

**De ce pragul 0,54 și nu 0,5?** Pragul se alege după obiectiv (recall ≥ 0,75 cu precizie maximă), pe train out-of-fold; 0,5 e arbitrar, mai ales cu ponderi pe clase. Și verificat după profit: optimul e 0,53.